# CAA2026 Classification Model

## Step 1: ENVIRONMENT SETUP

In [ ]:
# ============================================================
# ENVIRONMENT SETUP: Package Installation
# ============================================================
# This cell installs all required dependencies for the notebook.
# Run this first in a new environment (e.g., Google Colab, fresh conda env).
# ============================================================

# Cloud storage and data handling
# - gcsfs: Interface to Google Cloud Storage
# - zarr: Chunked, compressed N-dimensional arrays
# - pyarrow: Fast data interchange (for Parquet metadata)
# - huggingface_hub: Download pre-trained models/checkpoints
!pip -q install gcsfs zarr pyarrow huggingface_hub

# Machine learning utilities
# - scikit-learn: Evaluation metrics (AUC, precision-recall, etc.)
# - tqdm: Progress bars for loops
!pip -q install scikit-learn tqdm

print("✓ All packages installed successfully")

In [ ]:
# ============================================================
# ENVIRONMENT SETUP: Import Libraries
# ============================================================
# All package imports organized by category for clarity.
# ============================================================

# -----------------------------
# Standard Library
# -----------------------------
import os                    # File system operations
import re                    # Regular expressions
import gc                    # Garbage collection
import io                    # I/O operations
import json                  # JSON serialization
import math                  # Mathematical functions
import time                  # Time tracking
import shutil                # High-level file operations
import random                # Random number generation
import tarfile               # TAR archive handling
import subprocess            # Shell command execution
from pathlib import Path     # Object-oriented filesystem paths
from collections import Counter, defaultdict  # Specialized containers

# -----------------------------
# Data Science & Visualization
# -----------------------------
import numpy as np           # Numerical computing
import pandas as pd          # Tabular data manipulation
import matplotlib.pyplot as plt  # Plotting and visualization

# -----------------------------
# Cloud Storage & Data Formats
# -----------------------------
import zarr                  # Chunked array storage
import gcsfs                 # Google Cloud Storage filesystem interface

# -----------------------------
# Deep Learning (PyTorch)
# -----------------------------
import torch                 # Core PyTorch library
import torch.nn as nn        # Neural network modules
import torch.nn.functional as F  # Functional operations (activations, etc.)
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # Data handling

# -----------------------------
# Machine Learning Utilities
# -----------------------------
from sklearn.model_selection import train_test_split  # Data splitting
from sklearn.metrics import (
    roc_auc_score,           # Area under ROC curve
    average_precision_score, # Average precision (AP)
    precision_recall_curve,  # Precision-recall curve points
    confusion_matrix,        # Classification confusion matrix
    classification_report    # Detailed classification metrics
)

# -----------------------------
# Progress & Model Management
# -----------------------------
from tqdm.notebook import tqdm  # Notebook-friendly progress bars
from huggingface_hub import hf_hub_download  # Download models from HuggingFace Hub

## Step 2: CONFIGURATION

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# ============================================================================
# MODIFY THESE SETTINGS FOR YOUR ENVIRONMENT
# ============================================================================

# ----------------------------------------------------------------------------
# Google Cloud Storage Buckets
# ----------------------------------------------------------------------------
# Your GCS bucket names (without gs:// prefix)
# Example: 'my-project-data' (not 'gs://my-project-data')

# Bucket containing landcover-based negative samples (tar archives)
GCS_TAR_BUCKET = "YOUR-TAR-BUCKET-NAME"

# Bucket containing positive samples and integrated negatives (Zarr format)
GCS_ZARR_BUCKET = "YOUR-ZARR-BUCKET-NAME"

# ----------------------------------------------------------------------------
# Run Identification
# ----------------------------------------------------------------------------
# Name for this training run (used in checkpoint and report filenames)
# Example: 'experiment_v1'
STAGE1_RUN_NAME = "YOUR-RUN-NAME"

# ----------------------------------------------------------------------------
# Local Working Directories
# ----------------------------------------------------------------------------
# Base directory for all local data and outputs
# Default: '/content' (Google Colab)
# Modify if running locally or on other platforms (e.g., '/home/user/project')
LOCAL_BASE = "/content"

# ----------------------------------------------------------------------------
# Training Hyperparameters
# ----------------------------------------------------------------------------
BATCH_SIZE = 64              # Number of samples per training batch
NUM_WORKERS = 2              # Number of data loading workers
NUM_EPOCHS = 8               # Total training epochs
LEARNING_RATE = 1e-4         # Initial learning rate for Adam optimizer
WEIGHT_DECAY = 1e-4          # L2 regularization strength
USE_AMP = True               # Use Automatic Mixed Precision for faster training

# ----------------------------------------------------------------------------
# Data Split Ratios
# ----------------------------------------------------------------------------
TRAIN_FRAC = 0.70           # 70% for training
VAL_FRAC = 0.15             # 15% for validation
TEST_FRAC = 0.15            # 15% for testing (must sum to 1.0)

# ----------------------------------------------------------------------------
# Data Source Toggles
# ----------------------------------------------------------------------------
# Enable/disable different data sources for training
USE_POS_ZARR = True            # Use positive samples from Zarr
USE_INEG_ZARR = True           # Use integrated negative samples from Zarr
USE_LNEG_TAR = True            # Use landcover negative samples from tar archives
LOAD_CENTRAL_ASIA_EVAL = True  # Load Central Asia evaluation dataset

# Landcover negatives download options
DOWNLOAD_LNEG = True         # Download base landcover negative grids
DOWNLOAD_LNEG_AUG = False    # Download augmented versions (not needed; done online)

# ----------------------------------------------------------------------------
# Fast Run Mode (for testing/debugging)
# ----------------------------------------------------------------------------
FAST_RUN = False             # Set to True for quick testing with limited data
FAST_POS_LIMIT = None        # Limit positive samples (e.g., 100)
FAST_INEG_LIMIT = None       # Limit integrated negatives (e.g., 100)
FAST_LNEG_LIMIT = None       # Limit landcover negatives (e.g., 100)
FAST_CA_LIMIT = None         # Limit Central Asia samples (e.g., 50)


# ============================================================================
# DO NOT MODIFY BELOW THIS LINE (Auto-configured paths and settings)
# ============================================================================

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Device Configuration
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print("DEVICE CONFIGURATION")
print("=" * 70)
print(f"PyTorch version:    {torch.__version__}")
print(f"CUDA available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:         {torch.cuda.get_device_name(0)}")
print(f"Using device:       {device}")
print("=" * 70)

# -----------------------------
# GCS Paths (Auto-constructed)
# -----------------------------
# Tar-based landcover negatives
GCS_TAR_PACKS_PREFIX = f"gs://{GCS_TAR_BUCKET}/packs/"

# Zarr-based datasets
GCS_ZARR_ROOT = f"gs://{GCS_ZARR_BUCKET}/dataset/"
GCS_ZARR_POS = f"{GCS_ZARR_ROOT}positives.zarr"
GCS_ZARR_INEG = f"{GCS_ZARR_ROOT}integrated_negatives.zarr"
GCS_ZARR_META_POS = f"{GCS_ZARR_ROOT}metadata/positives.parquet"
GCS_ZARR_META_INEG = f"{GCS_ZARR_ROOT}metadata/integrated_negatives.parquet"

# -----------------------------
# Local Paths (Auto-constructed)
# -----------------------------
LOCAL_PACKS = f"{LOCAL_BASE}/packs"
LOCAL_GRIDS = f"{LOCAL_BASE}/grids"
LOCAL_ZARR_DIR = f"{LOCAL_BASE}/zarr_data"
LOCAL_META_DIR = f"{LOCAL_ZARR_DIR}/metadata"
LOCAL_POS_ZARR = f"{LOCAL_ZARR_DIR}/positives.zarr"
LOCAL_INEG_ZARR = f"{LOCAL_ZARR_DIR}/integrated_negatives.zarr"

LOCAL_CKPT = f"{LOCAL_BASE}/checkpoints"
LOCAL_REPORTS = f"{LOCAL_BASE}/reports"
LOCAL_SPLITS = f"{LOCAL_BASE}/splits"
LOCAL_DEBUG = f"{LOCAL_BASE}/debug"

LOCAL_CENTRAL_ASIA_DIR = f"{LOCAL_BASE}/hf_central_asia"
LOCAL_CENTRAL_ASIA_GRIDS = f"{LOCAL_CENTRAL_ASIA_DIR}/grid_images"

# -----------------------------
# Channel Specification
# -----------------------------
# Order of input channels (must match data preprocessing)
CHANNEL_ORDER = [
    "B2",    # Sentinel-2 Blue (490 nm)
    "B3",    # Sentinel-2 Green (560 nm)
    "B4",    # Sentinel-2 Red (665 nm)
    "B8",    # Sentinel-2 NIR (842 nm)
    "B11",   # Sentinel-2 SWIR1 (1610 nm)
    "B12",   # Sentinel-2 SWIR2 (2190 nm)
    "DEM",   # Digital Elevation Model (NASADEM)
    "Slope", # Terrain slope (derived from DEM)
    "NDVI",  # Normalized Difference Vegetation Index
    "NDWI",  # Normalized Difference Water Index
    "BSI"    # Bare Soil Index
]
N_CHANNELS = len(CHANNEL_ORDER)   # Total: 11 channels
IMAGE_SIZE = 100                  # Spatial dimensions: 100x100 pixels

# -----------------------------
# Output File Names
# -----------------------------
BEST_CKPT_NAME = f"{STAGE1_RUN_NAME}_best.pth"    # Best validation checkpoint
LAST_CKPT_NAME = f"{STAGE1_RUN_NAME}_last.pth"    # Final epoch checkpoint
TRAIN_STATS_JSON = f"{LOCAL_REPORTS}/{STAGE1_RUN_NAME}_train_channel_stats.json"

# -----------------------------
# Create Required Directories
# -----------------------------
for p in [
    LOCAL_PACKS, LOCAL_GRIDS, LOCAL_ZARR_DIR, LOCAL_META_DIR,
    LOCAL_CKPT, LOCAL_REPORTS, LOCAL_SPLITS, LOCAL_DEBUG,
    LOCAL_CENTRAL_ASIA_DIR
]:
    os.makedirs(p, exist_ok=True)

# -----------------------------
# Configuration Summary
# -----------------------------
print("\n" + "=" * 70)
print("USER CONFIGURATION LOADED")
print("=" * 70)
print(f"Run name:           {STAGE1_RUN_NAME}")
print(f"GCS Tar bucket:     {GCS_TAR_BUCKET}")
print(f"GCS Zarr bucket:    {GCS_ZARR_BUCKET}")
print(f"Local base:         {LOCAL_BASE}")
print("=" * 70)
print("\n⚠️  If you see 'YOUR-...' values above, you need to configure the top of this cell!")
print("=" * 70)
print("\nTRAINING CONFIGURATION")
print("=" * 70)
print(f"Batch size:         {BATCH_SIZE}")
print(f"Learning rate:      {LEARNING_RATE}")
print(f"Epochs:             {NUM_EPOCHS}")
print(f"Input channels:     {N_CHANNELS}")
print(f"Image size:         {IMAGE_SIZE}x{IMAGE_SIZE}")
print("=" * 70)
print("\nDATA SOURCES")
print("=" * 70)
print(f"Positive Zarr:      {USE_POS_ZARR}")
print(f"Integrated Neg:     {USE_INEG_ZARR}")
print(f"Landcover Neg:      {USE_LNEG_TAR}")
print(f"Central Asia Eval:  {LOAD_CENTRAL_ASIA_EVAL}")
print("=" * 70)
print("\nSTORAGE PATHS")
print("=" * 70)
print(f"GCS Tar bucket:     {GCS_TAR_PACKS_PREFIX}")
print(f"GCS Zarr root:      {GCS_ZARR_ROOT}")
print(f"Local checkpoints:  {LOCAL_CKPT}")
print(f"Local reports:      {LOCAL_REPORTS}")
print("=" * 70)
print("\n✓ Configuration complete")

## Step 3: DATA IMPORT / DOWNLOAD / LOCAL PREPARATION

In [ ]:
# ============================================================
# DATA IMPORT / DOWNLOAD / LOCAL PREPARATION
# ============================================================
# This section handles:
# 1. Google Cloud authentication
# 2. Downloading datasets from GCS (Zarr stores and tar archives)
# 3. Extracting and validating local data
# 4. Loading Central Asia evaluation dataset from Hugging Face
# ============================================================

# -----------------------------
# Authentication (Google Colab)
# -----------------------------
from google.colab import auth
auth.authenticate_user()
print("✓ Google authentication complete")


# -----------------------------
# Helper Functions
# -----------------------------

def run_cmd(cmd: str, check: bool = True):
    """
    Execute a shell command and print output.

    Args:
        cmd: Shell command to execute
        check: If True, raise RuntimeError on non-zero exit code

    Returns:
        subprocess.CompletedProcess result
    """
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed: {cmd}")
    return result


def gsutil_exists(gcs_path: str) -> bool:
    """Check if a GCS path exists."""
    result = subprocess.run(
        f'gsutil ls "{gcs_path}"',
        shell=True,
        text=True,
        capture_output=True
    )
    return result.returncode == 0


def download_gcs_file_or_dir(src: str, dst: str):
    """
    Download a file or directory from GCS to local filesystem.

    Args:
        src: GCS source path (gs://...)
        dst: Local destination path
    """
    # Determine if source is a directory (Zarr stores end with .zarr or /)
    is_dir = src.endswith(".zarr") or src.endswith("/")
    flag = "-r" if is_dir else ""
    run_cmd(f'gsutil -m cp {flag} "{src}" "{dst}"')


def list_tar_packs(prefix: str):
    """
    List all tar files in a GCS bucket prefix.

    Args:
        prefix: GCS path prefix (e.g., gs://bucket/packs/)

    Returns:
        List of tar filenames (sorted)
    """
    result = subprocess.run(
        f'gsutil ls "{prefix}"',
        shell=True,
        text=True,
        capture_output=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    packs = []
    for line in result.stdout.strip().splitlines():
        line = line.strip()
        if line.endswith(".tar"):
            packs.append(os.path.basename(line))
    return sorted(packs)


def select_lneg_tar_packs(all_packs, download_lneg=True, download_lneg_aug=False):
    """
    Filter tar packs to only landcover negatives based on config flags.

    Args:
        all_packs: List of all available tar filenames
        download_lneg: Include base landcover negatives (lneg_0*)
        download_lneg_aug: Include augmented landcover negatives (lneg_aug_*)

    Returns:
        Filtered list of tar filenames
    """
    prefixes = []
    if download_lneg:
        prefixes.append("lneg_0")
    if download_lneg_aug:
        prefixes.append("lneg_aug_")
    selected = [p for p in all_packs if any(p.startswith(pfx) for pfx in prefixes)]
    return sorted(selected)


def extract_tar_if_needed(tar_path: str, extract_root: str):
    """
    Extract tar archive, skipping directories that already exist locally.

    Args:
        tar_path: Path to tar file
        extract_root: Directory to extract into
    """
    with tarfile.open(tar_path, "r") as tar:
        # Get top-level directories in the tar
        top_dirs = {m.name.split("/")[0] for m in tar.getmembers() if "/" in m.name}

        # Get already-extracted directories
        existing = {d for d in os.listdir(extract_root) if os.path.isdir(os.path.join(extract_root, d))}

        # Only extract members from directories that don't exist yet
        members = [m for m in tar.getmembers() if m.name.split("/")[0] not in existing]

        if members:
            tar.extractall(extract_root, members=members)


def discover_local_grid_dirs(root: str):
    """
    Find all grid directories in a local folder.

    Args:
        root: Path to search

    Returns:
        Sorted list of directory names
    """
    if not os.path.exists(root):
        return []
    return sorted([
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    ])


def validate_landcover_grid_dir(grid_dir: str, channel_order):
    """
    Verify that a grid directory contains all required channel files and label.

    Args:
        grid_dir: Path to grid directory
        channel_order: List of channel names to check

    Returns:
        True if all required files exist, False otherwise
    """
    required = [f"channels/{c}.npy" for c in channel_order] + ["labels/binary_label.npy"]
    for rel in required:
        if not os.path.isfile(os.path.join(grid_dir, rel)):
            return False
    return True


def parse_grid_type(name: str):
    """
    Determine grid type from directory name prefix.

    Args:
        name: Grid directory name

    Returns:
        'pos', 'ineg', 'lneg', 'unla', or 'unknown'
    """
    if name.startswith("grid_"):
        return "pos"
    if name.startswith("ineg_"):
        return "ineg"
    if name.startswith("lneg_"):
        return "lneg"
    if name.startswith("unla_"):
        return "unla"
    return "unknown"


def maybe_limit_list(items, limit):
    """
    Optionally limit a list to first N items (for FAST_RUN mode).

    Args:
        items: List to potentially limit
        limit: Maximum number of items (None = no limit)

    Returns:
        Original or truncated list
    """
    if limit is None:
        return items
    return items[:limit]


# -----------------------------
# Download Zarr Stores + Metadata
# -----------------------------

# Positive samples (archaeological sites)
if USE_POS_ZARR:
    if os.path.exists(LOCAL_POS_ZARR):
        print(f"✓ positives.zarr already local: {LOCAL_POS_ZARR}")
    else:
        print("Downloading positives.zarr ...")
        t0 = time.time()
        download_gcs_file_or_dir(GCS_ZARR_POS, LOCAL_ZARR_DIR)
        print(f"✓ positives.zarr downloaded in {time.time() - t0:.1f}s")

# Integrated negatives (non-site samples)
if USE_INEG_ZARR:
    if os.path.exists(LOCAL_INEG_ZARR):
        print(f"✓ integrated_negatives.zarr already local: {LOCAL_INEG_ZARR}")
    else:
        print("Downloading integrated_negatives.zarr ...")
        t0 = time.time()
        download_gcs_file_or_dir(GCS_ZARR_INEG, LOCAL_ZARR_DIR)
        print(f"✓ integrated_negatives.zarr downloaded in {time.time() - t0:.1f}s")

# Download metadata files (Parquet format)
for src, dst in [
    (GCS_ZARR_META_POS, f"{LOCAL_META_DIR}/positives.parquet"),
    (GCS_ZARR_META_INEG, f"{LOCAL_META_DIR}/integrated_negatives.parquet"),
]:
    if os.path.exists(dst):
        print(f"✓ already local: {dst}")
    else:
        print(f"Downloading metadata: {src}")
        t0 = time.time()
        download_gcs_file_or_dir(src, dst)
        print(f"✓ metadata downloaded in {time.time() - t0:.1f}s")


# -----------------------------
# Download Landcover-Negative Tar Packs and Extract
# -----------------------------
local_lneg_dirs = []

if USE_LNEG_TAR:
    print("\nListing tar packs from GCS ...")
    all_packs = list_tar_packs(GCS_TAR_PACKS_PREFIX)
    lneg_packs = select_lneg_tar_packs(
        all_packs,
        download_lneg=DOWNLOAD_LNEG,
        download_lneg_aug=DOWNLOAD_LNEG_AUG
    )

    print(f"Total tar packs found: {len(all_packs)}")
    print(f"Selected landcover tar packs: {len(lneg_packs)}")

    # Download only packs that aren't already local
    already_local = {f for f in os.listdir(LOCAL_PACKS) if f.endswith(".tar")}
    to_download = [p for p in lneg_packs if p not in already_local]

    print(f"Need to download: {len(to_download)}")
    if to_download:
        t0 = time.time()
        sources = [f"{GCS_TAR_PACKS_PREFIX}{p}" for p in to_download]
        quoted_sources = " ".join([f'"{s}"' for s in sources])
        run_cmd(f'gsutil -m cp {quoted_sources} "{LOCAL_PACKS}/"')
        print(f"✓ tar download finished in {time.time() - t0:.1f}s")

    # Get list of downloaded tar files
    tar_files = sorted([
        os.path.join(LOCAL_PACKS, f)
        for f in os.listdir(LOCAL_PACKS)
        if f.endswith(".tar") and f in set(lneg_packs)
    ])

    # Extract tar archives (skips already-extracted directories)
    print(f"\nExtracting selected landcover tar packs: {len(tar_files)}")
    t0 = time.time()
    for tar_path in tqdm(tar_files, desc="Extract tar"):
        extract_tar_if_needed(tar_path, LOCAL_GRIDS)
    print(f"✓ extraction finished in {time.time() - t0:.1f}s")

    # Discover and validate landcover-negative grid directories
    all_local_dirs = discover_local_grid_dirs(LOCAL_GRIDS)
    local_lneg_dirs = [
        d for d in all_local_dirs
        if d.startswith("lneg_") and validate_landcover_grid_dir(os.path.join(LOCAL_GRIDS, d), CHANNEL_ORDER)
    ]

    # Apply FAST_RUN limit if enabled
    local_lneg_dirs = maybe_limit_list(local_lneg_dirs, FAST_LNEG_LIMIT if FAST_RUN else None)

    print(f"✓ valid local landcover-negative grid dirs: {len(local_lneg_dirs)}")


# -----------------------------
# Open Zarr Stores + Load Metadata
# -----------------------------
pos_root = None
ineg_root = None
pos_meta = None
ineg_meta = None

if USE_POS_ZARR:
    pos_root = zarr.open(LOCAL_POS_ZARR, mode="r")
    pos_meta_path = f"{LOCAL_META_DIR}/positives.parquet"
    pos_meta = pd.read_parquet(pos_meta_path)
    print(f"\n✓ positives.zarr opened")
    print(f"  metadata rows: {len(pos_meta):,}")
    print(f"  zarr keys: {list(pos_root.array_keys()) if hasattr(pos_root, 'array_keys') else 'N/A'}")

if USE_INEG_ZARR:
    ineg_root = zarr.open(LOCAL_INEG_ZARR, mode="r")
    ineg_meta_path = f"{LOCAL_META_DIR}/integrated_negatives.parquet"
    ineg_meta = pd.read_parquet(ineg_meta_path)
    print(f"\n✓ integrated_negatives.zarr opened")
    print(f"  metadata rows: {len(ineg_meta):,}")
    print(f"  zarr keys: {list(ineg_root.array_keys()) if hasattr(ineg_root, 'array_keys') else 'N/A'}")


# -----------------------------
# Download Central Asia Dataset for External Evaluation
# -----------------------------
# This dataset from Hugging Face is used for transfer learning evaluation
central_asia_grid_dirs = []

if LOAD_CENTRAL_ASIA_EVAL:
    HF_DATASET = "lldbrett/archaeological-sites-central-asia"

    # Download RAR file from Hugging Face if not already present
    rar_files = [f for f in os.listdir(LOCAL_CENTRAL_ASIA_DIR) if f.lower().endswith(".rar")]
    if rar_files:
        rar_path = os.path.join(LOCAL_CENTRAL_ASIA_DIR, rar_files[0])
        print(f"\n✓ Central Asia rar already local: {rar_path}")
    else:
        print("\nDownloading CentralAsia.rar from Hugging Face ...")
        rar_path = hf_hub_download(
            repo_id=HF_DATASET,
            filename="CentralAsia.rar",
            repo_type="dataset",
            local_dir=LOCAL_CENTRAL_ASIA_DIR
        )
        print(f"✓ Downloaded: {rar_path}")

    # Install unrar utility and extract archive
    if not os.path.exists(LOCAL_CENTRAL_ASIA_GRIDS):
        print("Extracting CentralAsia.rar (this may take a moment) ...")
        run_cmd("apt-get install -qq unrar")
        result = subprocess.run(
            f'unrar x -inul -o+ "{rar_path}" "{LOCAL_CENTRAL_ASIA_DIR}/"',
            shell=True,
            text=True,
            capture_output=True
        )
        if result.returncode == 0:
            print("✓ Extraction complete")
        else:
            if result.stderr.strip():
                print(result.stderr.strip())
            raise RuntimeError("Extraction failed")
    else:
        print(f"✓ Central Asia already extracted: {LOCAL_CENTRAL_ASIA_GRIDS}")

    # Discover Central Asia grid directories
    if os.path.exists(LOCAL_CENTRAL_ASIA_GRIDS):
        central_asia_grid_dirs = sorted([
            d for d in os.listdir(LOCAL_CENTRAL_ASIA_GRIDS)
            if os.path.isdir(os.path.join(LOCAL_CENTRAL_ASIA_GRIDS, d))
        ])

        # Apply FAST_RUN limit if enabled
        if FAST_RUN:
            central_asia_grid_dirs = maybe_limit_list(central_asia_grid_dirs, FAST_CA_LIMIT)

        print(f"✓ Central Asia grid dirs found: {len(central_asia_grid_dirs)}")


# -----------------------------
# Data Import Summary
# -----------------------------
print("\n" + "=" * 70)
print("DATA IMPORT SUMMARY")
print("=" * 70)

print(f"Positives zarr loaded:            {pos_root is not None}")
print(f"Integrated negatives zarr loaded: {ineg_root is not None}")
print(f"Landcover negatives local dirs:   {len(local_lneg_dirs):,}")
print(f"Central Asia eval dirs:           {len(central_asia_grid_dirs):,}")

if pos_meta is not None:
    print(f"Positive metadata rows:           {len(pos_meta):,}")
if ineg_meta is not None:
    print(f"Integrated-negative metadata rows:{len(ineg_meta):,}")

print("\nSample landcover dirs:")
print(local_lneg_dirs[:5])

print("\nSample Central Asia dirs:")
print(central_asia_grid_dirs[:5])

print("\n✓ Data import complete")

## Step 4: BUILD UNIFIED SAMPLE INDEX

In [ ]:
# ============================================================
# BUILD UNIFIED SAMPLE INDEX
# ============================================================
# This section creates a unified catalog of all training samples from
# different data sources (positive/negative Zarr, landcover tar archives,
# Central Asia evaluation set). Each sample gets metadata including:
# - Unique ID, label, region, storage location
# - Group ID (for splitting data while keeping related samples together)
# - Geographic coordinates when available
# ============================================================

# -----------------------------
# Helper Functions
# -----------------------------

def safe_int_from_match(pattern: str, text: str, default=None):
    """
    Extract integer from regex match or return default.

    Args:
        pattern: Regex pattern with one capture group
        text: String to search
        default: Value to return if no match

    Returns:
        Extracted integer or default value
    """
    m = re.search(pattern, text)
    return int(m.group(1)) if m else default


def infer_pos_group_id(row):
    """
    Group positives by underlying site_id so all variants from the same
    archaeological site stay in the same train/val/test split.

    This ensures that different views, rotations, or augmentations of
    the same site don't leak across splits.

    Args:
        row: DataFrame row with site_id and grid_id columns

    Returns:
        Group identifier string (e.g., "site::12345")
    """
    site_id = str(row.get("site_id", "")).strip()
    if site_id:
        return f"site::{site_id}"
    grid_id = str(row.get("grid_id", "")).strip()
    return f"pos_grid::{grid_id}"


def infer_ineg_group_id(row):
    """
    Group integrated negatives by their source site identifier, so they
    stay with the corresponding positive site during splitting.

    Integrated negatives are non-site samples from near known sites,
    so they should remain in the same split as their parent site.

    Args:
        row: DataFrame row with source_site_id and grid_id columns

    Returns:
        Group identifier string (e.g., "site::12345")
    """
    source_site_id = str(row.get("source_site_id", "")).strip()
    if source_site_id:
        return f"site::{source_site_id}"
    grid_id = str(row.get("grid_id", "")).strip()
    return f"ineg_grid::{grid_id}"


def parse_lneg_numeric_id(grid_dir_name: str):
    """
    Extract numeric ID from landcover negative directory name.
    Example: "lneg_00123" -> "00123"

    Args:
        grid_dir_name: Directory name (e.g., "lneg_00123")

    Returns:
        Numeric ID string or original name if no match
    """
    m = re.match(r"lneg_(\d+)", grid_dir_name)
    return m.group(1) if m else grid_dir_name


def build_positive_index(pos_meta: pd.DataFrame):
    """
    Build index for positive samples (archaeological sites) from Zarr metadata.

    Args:
        pos_meta: Metadata DataFrame from positives.parquet

    Returns:
        Standardized DataFrame with sample metadata
    """
    df = pos_meta.copy()

    # Core identification
    df["sample_id"] = df["grid_id"].astype(str)
    df["class_label"] = 1  # All positives are class 1
    df["region"] = "amazon"
    df["source_family"] = "positive_zarr"
    df["storage_type"] = "zarr"
    df["group_id"] = df.apply(infer_pos_group_id, axis=1)

    # Spatial augmentation metadata
    df["position"] = df.get("position", pd.Series(["unknown"] * len(df))).astype(str)
    df["rotation"] = pd.to_numeric(df.get("rotation", 0), errors="coerce").fillna(0).astype(int)

    # Storage location
    df["path_or_store"] = LOCAL_POS_ZARR
    df["local_path"] = None
    df["zarr_index"] = pd.to_numeric(df["zarr_index"], errors="coerce").astype(int)

    # Normalize column names across data sources
    if "site_id" not in df.columns:
        df["site_id"] = None
    if "site_type" not in df.columns:
        df["site_type"] = None

    # Standardized column set
    keep_cols = [
        "sample_id", "class_label", "region", "source_family", "storage_type",
        "group_id", "site_id", "site_type", "grid_id", "position", "rotation",
        "lat", "lon", "path_or_store", "local_path", "zarr_index"
    ]
    return df[keep_cols].copy()


def build_ineg_index(ineg_meta: pd.DataFrame):
    """
    Build index for integrated negative samples from Zarr metadata.

    Integrated negatives are non-site samples from contextual areas
    around known archaeological sites.

    Args:
        ineg_meta: Metadata DataFrame from integrated_negatives.parquet

    Returns:
        Standardized DataFrame with sample metadata
    """
    df = ineg_meta.copy()

    # Core identification
    df["sample_id"] = df["grid_id"].astype(str)
    df["class_label"] = 0  # All integrated negatives are class 0
    df["region"] = "amazon"
    df["source_family"] = "integrated_negative_zarr"
    df["storage_type"] = "zarr"
    df["group_id"] = df.apply(infer_ineg_group_id, axis=1)

    # Spatial metadata
    df["position"] = "integrated_context"
    df["rotation"] = pd.to_numeric(df.get("rotation", 0), errors="coerce").fillna(0).astype(int)

    # Storage location
    df["path_or_store"] = LOCAL_INEG_ZARR
    df["local_path"] = None
    df["zarr_index"] = pd.to_numeric(df["zarr_index"], errors="coerce").astype(int)

    # Normalize naming: link to source site if available
    if "source_site_id" in df.columns:
        df["site_id"] = df["source_site_id"].astype(str)
    else:
        df["site_id"] = None

    if "neg_type" not in df.columns:
        df["neg_type"] = None

    df["site_type"] = df.get("neg_type", None)

    # Standardized column set
    keep_cols = [
        "sample_id", "class_label", "region", "source_family", "storage_type",
        "group_id", "site_id", "site_type", "grid_id", "position", "rotation",
        "lat", "lon", "path_or_store", "local_path", "zarr_index"
    ]
    return df[keep_cols].copy()


def build_lneg_index(local_lneg_dirs, local_grids_root):
    """
    Build index for landcover-based negative samples from tar archives.

    These are negative samples extracted from various land cover types
    (forest, water, agriculture, etc.) to ensure the model doesn't just
    learn archaeological site patterns but also what is NOT a site.

    Args:
        local_lneg_dirs: List of landcover negative directory names
        local_grids_root: Root directory containing extracted grids

    Returns:
        DataFrame with sample metadata
    """
    rows = []

    for dname in tqdm(local_lneg_dirs, desc="Index lneg tar grids"):
        grid_path = os.path.join(local_grids_root, dname)

        # Validate that all required files exist
        if not validate_landcover_grid_dir(grid_path, CHANNEL_ORDER):
            continue

        numeric_id = parse_lneg_numeric_id(dname)

        rows.append({
            "sample_id": dname,
            "class_label": 0,  # Landcover negatives are class 0
            "region": "amazon",
            "source_family": "landcover_negative_tar",
            "storage_type": "tar_local",
            "group_id": f"lneg::{numeric_id}",
            "site_id": None,
            "site_type": "landcover_negative",
            "grid_id": dname,
            "position": "landcover",
            "rotation": 0,
            "lat": None,
            "lon": None,
            "path_or_store": LOCAL_GRIDS,
            "local_path": grid_path,
            "zarr_index": None
        })

    return pd.DataFrame(rows)


def parse_ca_group_id(grid_name: str):
    """
    Parse Central Asia grid name to extract base site identifier.

    Examples:
      - grid_000001_rot000 -> "ca::grid_000001"
      - grid_000001_rot000_aug1 -> "ca::grid_000001"
      - grid_000001_rot120 -> "ca::grid_000001"

    Groups all rotations/augmentations of the same site together
    so they stay in the same train/val/test split.

    Args:
        grid_name: Directory name from Central Asia dataset

    Returns:
        Group identifier (e.g., "ca::grid_000001")
    """
    m = re.match(r"(grid_\d+)_rot\d+(?:_aug\d+)?$", grid_name)
    if m:
        return f"ca::{m.group(1)}"
    return f"ca::{grid_name}"


def parse_ca_rotation(grid_name: str):
    """Extract rotation angle from Central Asia grid name."""
    return safe_int_from_match(r"_rot(\d+)", grid_name, default=0)


def parse_ca_aug(grid_name: str):
    """Extract augmentation ID from Central Asia grid name."""
    return safe_int_from_match(r"_aug(\d+)", grid_name, default=0)


def build_central_asia_index(central_asia_grid_dirs, local_ca_root):
    """
    Build index for Central Asia evaluation dataset.

    This external dataset is used for transfer learning evaluation.
    Labels may not be available initially and are filled during QC.

    Args:
        central_asia_grid_dirs: List of Central Asia grid directory names
        local_ca_root: Root directory containing Central Asia grids

    Returns:
        DataFrame with sample metadata
    """
    rows = []

    for dname in tqdm(central_asia_grid_dirs, desc="Index Central Asia grids"):
        grid_path = os.path.join(local_ca_root, dname)
        if not os.path.isdir(grid_path):
            continue

        # Note: We don't require binary_label here yet, because external eval
        # structure may differ. Labels will be inspected during QC.
        rows.append({
            "sample_id": dname,
            "class_label": None,  # Fill later if labels are available
            "region": "central_asia",
            "source_family": "central_asia_external",
            "storage_type": "folder_local",
            "group_id": parse_ca_group_id(dname),
            "site_id": parse_ca_group_id(dname).replace("ca::", ""),
            "site_type": None,
            "grid_id": dname,
            "position": "external_eval",
            "rotation": parse_ca_rotation(dname),
            "aug_id": parse_ca_aug(dname),
            "lat": None,
            "lon": None,
            "path_or_store": local_ca_root,
            "local_path": grid_path,
            "zarr_index": None
        })

    return pd.DataFrame(rows)


# -----------------------------
# Build Component Indices
# -----------------------------

# Build index for each data source
pos_index_df = build_positive_index(pos_meta) if pos_meta is not None else pd.DataFrame()
ineg_index_df = build_ineg_index(ineg_meta) if ineg_meta is not None else pd.DataFrame()
lneg_index_df = build_lneg_index(local_lneg_dirs, LOCAL_GRIDS) if USE_LNEG_TAR else pd.DataFrame()
central_asia_samples_df = build_central_asia_index(
    central_asia_grid_dirs,
    LOCAL_CENTRAL_ASIA_GRIDS
) if LOAD_CENTRAL_ASIA_EVAL else pd.DataFrame()

# Apply FAST_RUN limits if enabled
if FAST_RUN:
    if FAST_POS_LIMIT is not None and len(pos_index_df):
        pos_index_df = pos_index_df.iloc[:FAST_POS_LIMIT].copy()
    if FAST_INEG_LIMIT is not None and len(ineg_index_df):
        ineg_index_df = ineg_index_df.iloc[:FAST_INEG_LIMIT].copy()
    if FAST_LNEG_LIMIT is not None and len(lneg_index_df):
        lneg_index_df = lneg_index_df.iloc[:FAST_LNEG_LIMIT].copy()
    if FAST_CA_LIMIT is not None and len(central_asia_samples_df):
        central_asia_samples_df = central_asia_samples_df.iloc[:FAST_CA_LIMIT].copy()


# -----------------------------
# Concatenate Amazon Training Data
# -----------------------------

# Combine all Amazon data sources (positives + both types of negatives)
amazon_parts = [df for df in [pos_index_df, ineg_index_df, lneg_index_df] if len(df) > 0]
amazon_samples_df = pd.concat(amazon_parts, axis=0, ignore_index=True)

# Enforce consistent column order for clarity
amazon_col_order = [
    "sample_id", "class_label", "region", "source_family", "storage_type",
    "group_id", "site_id", "site_type", "grid_id", "position", "rotation",
    "lat", "lon", "path_or_store", "local_path", "zarr_index"
]
amazon_samples_df = amazon_samples_df[amazon_col_order].copy()

# Central Asia has additional 'aug_id' column
if len(central_asia_samples_df):
    ca_col_order = [
        "sample_id", "class_label", "region", "source_family", "storage_type",
        "group_id", "site_id", "site_type", "grid_id", "position", "rotation",
        "aug_id", "lat", "lon", "path_or_store", "local_path", "zarr_index"
    ]
    central_asia_samples_df = central_asia_samples_df[ca_col_order].copy()


# -----------------------------
# Save Sample Index Artifacts
# -----------------------------

# Save indices to CSV for reproducibility and inspection
amazon_index_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_amazon_sample_index.csv"
central_asia_index_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_central_asia_sample_index.csv"

amazon_samples_df.to_csv(amazon_index_csv, index=False)
if len(central_asia_samples_df):
    central_asia_samples_df.to_csv(central_asia_index_csv, index=False)

print(f"✓ Saved Amazon sample index: {amazon_index_csv}")
if len(central_asia_samples_df):
    print(f"✓ Saved Central Asia sample index: {central_asia_index_csv}")


# -----------------------------
# Sample Index Summary
# -----------------------------

print("\n" + "=" * 70)
print("AMAZON SAMPLE INDEX SUMMARY")
print("=" * 70)
print(f"Total Amazon samples: {len(amazon_samples_df):,}")

print("\nBy source_family:")
print(amazon_samples_df["source_family"].value_counts(dropna=False))

print("\nBy class_label:")
print(amazon_samples_df["class_label"].value_counts(dropna=False))

print("\nUnique group_id count by source:")
print(
    amazon_samples_df.groupby("source_family")["group_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nHead:")
display(amazon_samples_df.head())

if len(central_asia_samples_df):
    print("\n" + "=" * 70)
    print("CENTRAL ASIA SAMPLE INDEX SUMMARY")
    print("=" * 70)
    print(f"Total Central Asia samples: {len(central_asia_samples_df):,}")
    print(f"\nUnique group_id: {central_asia_samples_df['group_id'].nunique()}")
    print("\nHead:")
    display(central_asia_samples_df.head())

print("\n✓ Sample index building complete")

## Step 5: DATA QUALITY CONTROL

In [ ]:
# ============================================================
# QC + LABEL DISCOVERY + GROUPED SPLIT
# ============================================================
# This section:
# 1. Performs quality control on local data folders
# 2. Discovers and validates classification labels
# 3. Creates train/val/test splits that prevent data leakage
#    by keeping related samples (same site, rotations, etc.) together
# ============================================================

# -----------------------------
# Helper Functions
# -----------------------------

def npy_exists(path):
    """Check if a .npy file exists."""
    return os.path.isfile(path)


def read_scalar_label_npy(path):
    """
    Read a scalar or single-element array from a .npy file.

    Args:
        path: Path to .npy file

    Returns:
        Integer value of the label
    """
    arr = np.load(path, allow_pickle=True)
    if np.isscalar(arr):
        return int(arr)
    arr = np.asarray(arr)
    if arr.size == 1:
        return int(arr.reshape(-1)[0])
    return arr


def inspect_local_grid_label(grid_dir):
    """
    Inspect a local grid directory to check for labels and data files.

    For tar/local folder samples, this verifies the directory structure
    and attempts to read the binary classification label.

    Args:
        grid_dir: Path to grid directory

    Returns:
        Dictionary with inspection results (label status, available files)
    """
    info = {
        "grid_dir": grid_dir,
        "has_binary_label": False,
        "binary_label_value": None,
        "has_labels_dir": False,
        "has_channels_dir": False,
        "available_label_files": [],
        "available_channel_files": [],
    }

    labels_dir = os.path.join(grid_dir, "labels")
    channels_dir = os.path.join(grid_dir, "channels")

    info["has_labels_dir"] = os.path.isdir(labels_dir)
    info["has_channels_dir"] = os.path.isdir(channels_dir)

    # Check for binary classification label
    if os.path.isdir(labels_dir):
        info["available_label_files"] = sorted(os.listdir(labels_dir))
        binary_label_path = os.path.join(labels_dir, "binary_label.npy")
        if os.path.isfile(binary_label_path):
            info["has_binary_label"] = True
            try:
                info["binary_label_value"] = int(read_scalar_label_npy(binary_label_path))
            except Exception as e:
                info["binary_label_value"] = f"ERROR: {e}"

    # List available channel files
    if os.path.isdir(channels_dir):
        info["available_channel_files"] = sorted(os.listdir(channels_dir))

    return info


def inspect_many_local_grids(grid_dirs, n=5):
    """
    Inspect multiple grid directories and return results as DataFrame.

    Args:
        grid_dirs: List of grid directory paths
        n: Number of grids to inspect

    Returns:
        DataFrame with inspection results
    """
    rows = []
    for gd in grid_dirs[:n]:
        rows.append(inspect_local_grid_label(gd))
    return pd.DataFrame(rows)


def grouped_train_val_test_split(df, group_col, stratify_col=None,
                                 train_frac=0.70, val_frac=0.15, test_frac=0.15,
                                 seed=42):
    """
    Split dataset by unique groups to prevent data leakage.

    This ensures that all samples from the same group (e.g., same archaeological
    site with different rotations/augmentations) stay in the same split.

    Stratification is approximate at the group level using the majority/first
    label from each group.

    Args:
        df: DataFrame with samples
        group_col: Column name for grouping (e.g., 'group_id')
        stratify_col: Column for stratification (defaults to 'class_label')
        train_frac: Fraction for training (default: 0.70)
        val_frac: Fraction for validation (default: 0.15)
        test_frac: Fraction for testing (default: 0.15)
        seed: Random seed for reproducibility

    Returns:
        DataFrame with added 'split' column
    """
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-8

    # Get one label per group for stratification
    group_df = (
        df.groupby(group_col, as_index=False)
        .agg({
            stratify_col if stratify_col else "class_label": "first"
        })
        .rename(columns={
            stratify_col if stratify_col else "class_label": "group_label"
        })
    )

    groups = group_df[group_col].values
    group_labels = group_df["group_label"].values

    # First split: train vs temp (val+test)
    train_groups, temp_groups, train_y, temp_y = train_test_split(
        groups,
        group_labels,
        test_size=(1.0 - train_frac),
        random_state=seed,
        stratify=group_labels
    )

    # Second split: val vs test within temp
    val_ratio_within_temp = val_frac / (val_frac + test_frac)
    val_groups, test_groups, _, _ = train_test_split(
        temp_groups,
        temp_y,
        test_size=(1.0 - val_ratio_within_temp),
        random_state=seed,
        stratify=temp_y
    )

    # Create mapping from group_id to split assignment
    split_map = {}
    for g in train_groups:
        split_map[g] = "train"
    for g in val_groups:
        split_map[g] = "val"
    for g in test_groups:
        split_map[g] = "test"

    # Apply split assignment to all samples
    out = df.copy()
    out["split"] = out[group_col].map(split_map)

    return out


def summarize_split(df, name):
    """
    Print detailed summary of data split distribution.

    Args:
        df: DataFrame with 'split', 'class_label', and 'source_family' columns
        name: Name to display in header
    """
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(f"Rows: {len(df):,}")
    print("\nBy split:")
    print(df["split"].value_counts(dropna=False))
    print("\nBy split x class:")
    print(pd.crosstab(df["split"], df["class_label"], dropna=False))
    print("\nBy split x source_family:")
    print(pd.crosstab(df["split"], df["source_family"], dropna=False))


# -----------------------------
# QC: Inspect Landcover Negative Local Folders
# -----------------------------

# Get paths to landcover negative samples
lneg_sample_paths = amazon_samples_df.loc[
    amazon_samples_df["source_family"] == "landcover_negative_tar", "local_path"
].dropna().tolist()

print("Inspecting a few landcover-negative folders ...")
lneg_inspect_df = inspect_many_local_grids(lneg_sample_paths, n=5)
display(lneg_inspect_df)

# Verify binary labels exist and are correct (should all be 0)
lneg_check_rows = []
for p in tqdm(lneg_sample_paths[:200], desc="QC lneg labels"):
    info = inspect_local_grid_label(p)
    lneg_check_rows.append({
        "grid_dir": os.path.basename(p),
        "has_binary_label": info["has_binary_label"],
        "binary_label_value": info["binary_label_value"],
    })

lneg_check_df = pd.DataFrame(lneg_check_rows)
print("\nLandcover-negative QC summary (first 200):")
display(lneg_check_df["has_binary_label"].value_counts(dropna=False))
display(lneg_check_df["binary_label_value"].value_counts(dropna=False).head(10))


# -----------------------------
# QC: Inspect Central Asia Folder Structure and Labels
# -----------------------------

# Get paths to Central Asia evaluation samples
ca_sample_paths = central_asia_samples_df["local_path"].dropna().tolist() if len(central_asia_samples_df) else []

print("\nInspecting a few Central Asia folders ...")
ca_inspect_df = inspect_many_local_grids(ca_sample_paths, n=8) if len(ca_sample_paths) else pd.DataFrame()
display(ca_inspect_df)

# Check Central Asia label availability
ca_check_rows = []
for p in tqdm(ca_sample_paths[:200], desc="QC Central Asia labels"):
    info = inspect_local_grid_label(p)
    ca_check_rows.append({
        "grid_dir": os.path.basename(p),
        "has_binary_label": info["has_binary_label"],
        "binary_label_value": info["binary_label_value"],
        "available_label_files": ",".join(info["available_label_files"][:10]) if info["available_label_files"] else "",
    })

ca_check_df = pd.DataFrame(ca_check_rows) if len(ca_check_rows) else pd.DataFrame()

if len(ca_check_df):
    print("\nCentral Asia QC summary (first 200):")
    display(ca_check_df["has_binary_label"].value_counts(dropna=False))
    display(ca_check_df["binary_label_value"].value_counts(dropna=False).head(20))
    display(ca_check_df.head(10))


# -----------------------------
# Fill/Verify Classification Labels for Local Folders
# -----------------------------

# For local folder sources (tar_local, folder_local), read the actual
# binary_label.npy file to verify or fill the class_label field
amazon_samples_df = amazon_samples_df.copy()


def verify_or_fill_local_class_label(row):
    """
    Read classification label from local folder if available.

    For Zarr-based samples, the label is already in metadata.
    For local folder samples, read from binary_label.npy file.

    Args:
        row: DataFrame row

    Returns:
        Verified or filled class label
    """
    # Only apply to local storage types
    if row["storage_type"] not in ["tar_local", "folder_local"]:
        return row["class_label"]

    lp = row["local_path"]
    if lp is None or not os.path.isdir(lp):
        return row["class_label"]

    binary_label_path = os.path.join(lp, "labels", "binary_label.npy")
    if os.path.isfile(binary_label_path):
        try:
            return int(read_scalar_label_npy(binary_label_path))
        except:
            return row["class_label"]
    return row["class_label"]


amazon_samples_df["class_label_checked"] = amazon_samples_df.apply(verify_or_fill_local_class_label, axis=1)

# Sanity-check: landcover negatives should all be class 0
lneg_rows = amazon_samples_df["source_family"] == "landcover_negative_tar"
lneg_mismatch = amazon_samples_df.loc[lneg_rows & (amazon_samples_df["class_label_checked"] != 0)]

print("\nLandcover-negative mismatches after checking binary_label.npy:")
print(len(lneg_mismatch))

# Commit verified labels
amazon_samples_df["class_label"] = amazon_samples_df["class_label_checked"].astype(int)
amazon_samples_df.drop(columns=["class_label_checked"], inplace=True)

# Verify/fill Central Asia labels
if len(central_asia_samples_df):
    central_asia_samples_df = central_asia_samples_df.copy()
    central_asia_samples_df["class_label_checked"] = central_asia_samples_df.apply(verify_or_fill_local_class_label, axis=1)
    central_asia_samples_df["class_label"] = central_asia_samples_df["class_label_checked"]
    central_asia_samples_df.drop(columns=["class_label_checked"], inplace=True)

    ca_labeled = central_asia_samples_df["class_label"].notna().sum()
    print(f"\nCentral Asia labeled rows discovered: {ca_labeled:,} / {len(central_asia_samples_df):,}")


# -----------------------------
# Grouped Split for Amazon Training Data
# -----------------------------

# Create train/val/test splits while keeping groups together
# This prevents data leakage where different views of the same site
# appear in multiple splits
amazon_split_df = grouped_train_val_test_split(
    amazon_samples_df,
    group_col="group_id",
    stratify_col="class_label",
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SEED
)

summarize_split(amazon_split_df, "AMAZON GROUPED SPLIT SUMMARY")

# Verify no data leakage: each group should appear in exactly one split
group_split_counts = amazon_split_df.groupby("group_id")["split"].nunique()
leaky_groups = (group_split_counts > 1).sum()
print(f"\nLeakage check — groups appearing in >1 split: {leaky_groups}")
if leaky_groups > 0:
    print("⚠️  WARNING: Data leakage detected! Some groups span multiple splits.")
else:
    print("✓ No data leakage: all groups are isolated to single splits")


# -----------------------------
# Save Split Artifacts
# -----------------------------

# Save complete split and individual split CSVs for easy loading
amazon_split_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_amazon_split.csv"
amazon_train_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_train.csv"
amazon_val_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_val.csv"
amazon_test_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_test.csv"
central_asia_eval_csv = f"{LOCAL_SPLITS}/{STAGE1_RUN_NAME}_central_asia_eval.csv"

amazon_split_df.to_csv(amazon_split_csv, index=False)
amazon_split_df.loc[amazon_split_df["split"] == "train"].to_csv(amazon_train_csv, index=False)
amazon_split_df.loc[amazon_split_df["split"] == "val"].to_csv(amazon_val_csv, index=False)
amazon_split_df.loc[amazon_split_df["split"] == "test"].to_csv(amazon_test_csv, index=False)

if len(central_asia_samples_df):
    central_asia_samples_df.to_csv(central_asia_eval_csv, index=False)

print(f"\n✓ Saved: {amazon_split_csv}")
print(f"✓ Saved: {amazon_train_csv}")
print(f"✓ Saved: {amazon_val_csv}")
print(f"✓ Saved: {amazon_test_csv}")
if len(central_asia_samples_df):
    print(f"✓ Saved: {central_asia_eval_csv}")

print("\n✓ QC and split generation complete")

## Step 6: CHANNEL STATISTICS & NORMALIZATION

In [ ]:
# ============================================================
# CHANNEL STATISTICS & NORMALIZATION
# ============================================================
# This section:
# 1. Computes channel-wise statistics (mean, std, percentiles) from TRAIN set
# 2. Defines physical unit conversions (raw sensor values -> reflectance/elevation)
# 3. Creates normalization parameters for z-score standardization
# 4. Provides normalization function for consistent preprocessing
#
# CRITICAL: All statistics are computed from TRAINING data only to prevent
# data leakage. These same stats will be used to normalize train/val/test/eval.
# ============================================================

# Load training split for statistics computation
TRAIN_DF = pd.read_csv(amazon_train_csv)

# -----------------------------
# Detect Zarr Array Structure
# -----------------------------

def list_group_members(zroot):
    """List top-level keys in a Zarr group."""
    try:
        return list(zroot.keys())
    except:
        return []


print("Positive zarr members:", list_group_members(pos_root))
print("Integrated negative zarr members:", list_group_members(ineg_root))


def detect_zarr_image_array(zroot):
    """
    Auto-detect the main image array in a Zarr group.

    Tries common naming conventions: 'images', 'x', 'data', 'X'

    Args:
        zroot: Zarr group

    Returns:
        Key name of the image array

    Raises:
        KeyError: If no suitable array is found
    """
    candidate_names = ["images", "x", "data", "X"]
    for name in candidate_names:
        if name in zroot:
            arr = zroot[name]
            if hasattr(arr, "shape") and len(arr.shape) >= 3:
                return name

    # Fallback: first array-like child with 3+ dimensions
    for k in zroot.keys():
        obj = zroot[k]
        if hasattr(obj, "shape") and len(obj.shape) >= 3:
            return k

    raise KeyError("Could not find image array inside zarr group.")


# For this dataset, the arrays are named "grids"
POS_IMAGE_ARRAY_KEY = "grids" if pos_root is not None else None
INEG_IMAGE_ARRAY_KEY = "grids" if ineg_root is not None else None

pos_img_arr = pos_root[POS_IMAGE_ARRAY_KEY] if POS_IMAGE_ARRAY_KEY is not None else None
ineg_img_arr = ineg_root[INEG_IMAGE_ARRAY_KEY] if INEG_IMAGE_ARRAY_KEY is not None else None

print("Detected positive image array key:", POS_IMAGE_ARRAY_KEY)
print("Detected integrated-negative image array key:", INEG_IMAGE_ARRAY_KEY)

if pos_img_arr is not None:
    print("Positive image array shape:", pos_img_arr.shape, "dtype:", pos_img_arr.dtype)
if ineg_img_arr is not None:
    print("Integrated-negative image array shape:", ineg_img_arr.shape, "dtype:", ineg_img_arr.dtype)


# -----------------------------
# Raw Data Readers
# -----------------------------

def read_tar_grid_raw(local_grid_path, channel_order=CHANNEL_ORDER):
    """
    Read a grid from local tar-extracted folder structure.

    Args:
        local_grid_path: Path to grid directory
        channel_order: Ordered list of channel names

    Returns:
        Array with shape (C, H, W), dtype float32
    """
    chans = []
    for c in channel_order:
        arr = np.load(os.path.join(local_grid_path, "channels", f"{c}.npy"))
        chans.append(arr.astype(np.float32))
    x = np.stack(chans, axis=0)  # (C, H, W)
    return x


def read_zarr_raw(row, pos_img_arr, ineg_img_arr):
    """
    Read zarr sample by row from unified sample table.

    Expected layout:
      positives.zarr["grids"]              -> (N, 11, 100, 100)
      integrated_negatives.zarr["grids"]   -> (N, 11, 100, 100)

    Args:
        row: DataFrame row with zarr_index and source_family
        pos_img_arr: Positive zarr array
        ineg_img_arr: Integrated negative zarr array

    Returns:
        Array with shape (C, H, W), dtype float32
    """
    idx = int(row["zarr_index"])
    src = row["source_family"]

    if src == "positive_zarr":
        x = np.asarray(pos_img_arr[idx]).astype(np.float32)
    elif src == "integrated_negative_zarr":
        x = np.asarray(ineg_img_arr[idx]).astype(np.float32)
    else:
        raise ValueError(f"Unsupported zarr source_family: {src}")

    if x.ndim != 3:
        raise ValueError(f"Expected 3D sample array, got shape={x.shape}")

    # Expected format: (C, H, W)
    if x.shape[0] == N_CHANNELS:
        return x

    # Fallback: if stored as (H, W, C), transpose to (C, H, W)
    if x.shape[-1] == N_CHANNELS:
        return np.transpose(x, (2, 0, 1))

    raise ValueError(f"Cannot infer channel axis for shape={x.shape}")


def read_sample_raw_from_row(row):
    """
    Universal reader that dispatches based on storage type.

    Args:
        row: DataFrame row with storage_type and path information

    Returns:
        Raw array with shape (C, H, W), dtype float32
    """
    st = row["storage_type"]
    if st == "zarr":
        return read_zarr_raw(row, pos_img_arr, ineg_img_arr)
    elif st in ["tar_local", "folder_local"]:
        return read_tar_grid_raw(row["local_path"], CHANNEL_ORDER)
    else:
        raise ValueError(f"Unknown storage_type: {st}")


# -----------------------------
# Physical Unit Conversion
# -----------------------------

# Channel indices by type
OPTICAL_IDX = [0, 1, 2, 3, 4, 5]   # Sentinel-2 bands (B2, B3, B4, B8, B11, B12)
DEM_IDX = [6]                      # Digital Elevation Model
SLOPE_IDX = [7]                    # Terrain slope
INDEX_IDX = [8, 9, 10]             # Spectral indices (NDVI, NDWI, BSI)


def raw_to_physical(x_raw):
    """
    Convert raw sensor/stored values to physical units.

    Sentinel-2 scaling:
      - Optical bands: stored as DN (digital numbers), divide by 10000 to get reflectance [0-1]
      - Spectral indices: stored scaled, divide by 10000 (preserves negative values)

    DEM/Slope:
      - Already in physical units (meters, degrees), no scaling needed

    Args:
        x_raw: Raw array with shape (C, H, W), dtype float32

    Returns:
        Physical units array with shape (C, H, W), dtype float32
    """
    x = x_raw.astype(np.float32).copy()

    x_phys = np.zeros_like(x, dtype=np.float32)

    # Optical bands: DN -> reflectance (divide by 10000)
    x_phys[OPTICAL_IDX] = x[OPTICAL_IDX] / 10000.0

    # DEM: already in meters, no scaling
    x_phys[DEM_IDX] = x[DEM_IDX]

    # Slope: already in degrees, no scaling
    x_phys[SLOPE_IDX] = x[SLOPE_IDX]

    # Spectral indices: descale from stored values
    x_phys[INDEX_IDX] = x[INDEX_IDX] / 10000.0

    return x_phys


# -----------------------------
# Sample Training Data for Statistics
# -----------------------------

# Use all train rows if manageable; otherwise sample strategically
MAX_TRAIN_STATS_SAMPLES = 3000

if len(TRAIN_DF) <= MAX_TRAIN_STATS_SAMPLES:
    train_stats_df = TRAIN_DF.copy()
else:
    # Stratified sample by source + class for better coverage of all data types
    train_stats_df = (
        TRAIN_DF.groupby(["source_family", "class_label"], group_keys=False)
        .apply(lambda g: g.sample(
            n=max(1, int(round(MAX_TRAIN_STATS_SAMPLES * len(g) / len(TRAIN_DF)))),
            random_state=SEED
        ))
        .reset_index(drop=True)
    )

print(f"Train rows total: {len(TRAIN_DF):,}")
print(f"Rows used for stats: {len(train_stats_df):,}")
print(train_stats_df["source_family"].value_counts())
print(train_stats_df["class_label"].value_counts())


# -----------------------------
# Channel Statistics Collection
# -----------------------------

def init_channel_value_lists():
    """Initialize empty lists to collect values for each channel."""
    return {ch: [] for ch in CHANNEL_ORDER}


# Collectors for raw and physical values
raw_channel_values = init_channel_value_lists()
phys_channel_values = init_channel_value_lists()

# Optional: separate by class for quality control
phys_channel_values_pos = init_channel_value_lists()
phys_channel_values_neg = init_channel_value_lists()

# Subsample pixels per image to reduce memory usage
PIXELS_PER_IMAGE_FOR_STATS = 2048


def sample_pixels_1d(arr2d, k=2048, rng=None):
    """
    Randomly sample k pixels from a 2D array.

    Args:
        arr2d: 2D numpy array
        k: Number of pixels to sample
        rng: Numpy random number generator

    Returns:
        1D array of sampled pixel values
    """
    flat = arr2d.reshape(-1)
    n = flat.shape[0]
    if n <= k:
        return flat
    idx = rng.choice(n, size=k, replace=False)
    return flat[idx]


rng = np.random.default_rng(SEED)

# Collect statistics from training samples
for _, row in tqdm(train_stats_df.iterrows(), total=len(train_stats_df), desc="Collect channel stats"):
    x_raw = read_sample_raw_from_row(row)
    x_phys = raw_to_physical(x_raw)

    # Sanity check dimensions
    assert x_raw.shape[0] == N_CHANNELS, f"Unexpected raw shape: {x_raw.shape}"
    assert x_phys.shape[0] == N_CHANNELS, f"Unexpected phys shape: {x_phys.shape}"

    y = int(row["class_label"])

    # Sample pixels from each channel
    for ci, ch in enumerate(CHANNEL_ORDER):
        raw_vals = sample_pixels_1d(x_raw[ci], k=PIXELS_PER_IMAGE_FOR_STATS, rng=rng)
        phys_vals = sample_pixels_1d(x_phys[ci], k=PIXELS_PER_IMAGE_FOR_STATS, rng=rng)

        raw_channel_values[ch].append(raw_vals)
        phys_channel_values[ch].append(phys_vals)

        # Separate by class for QC analysis
        if y == 1:
            phys_channel_values_pos[ch].append(phys_vals)
        else:
            phys_channel_values_neg[ch].append(phys_vals)


def concat_channel_lists(d):
    """Concatenate collected pixel value lists into arrays."""
    out = {}
    for ch, parts in d.items():
        if len(parts) == 0:
            out[ch] = np.array([], dtype=np.float32)
        else:
            out[ch] = np.concatenate(parts).astype(np.float32)
    return out


# Concatenate all collected values
raw_channel_values = concat_channel_lists(raw_channel_values)
phys_channel_values = concat_channel_lists(phys_channel_values)
phys_channel_values_pos = concat_channel_lists(phys_channel_values_pos)
phys_channel_values_neg = concat_channel_lists(phys_channel_values_neg)


# -----------------------------
# Compute Summary Statistics
# -----------------------------

def summarize_1d(arr):
    """
    Compute summary statistics for a 1D array.

    Args:
        arr: 1D numpy array

    Returns:
        Dictionary with count, min, max, mean, std, and percentiles
    """
    arr = np.asarray(arr)
    if arr.size == 0:
        return {
            "count": 0,
            "min": np.nan, "max": np.nan,
            "mean": np.nan, "std": np.nan,
            "p1": np.nan, "p5": np.nan, "p50": np.nan, "p95": np.nan, "p99": np.nan
        }
    return {
        "count": int(arr.size),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "p1": float(np.percentile(arr, 1)),
        "p5": float(np.percentile(arr, 5)),
        "p50": float(np.percentile(arr, 50)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99)),
    }


# Compute statistics for each channel
rows = []
for ch in CHANNEL_ORDER:
    raw_s = summarize_1d(raw_channel_values[ch])
    phys_s = summarize_1d(phys_channel_values[ch])
    pos_s = summarize_1d(phys_channel_values_pos[ch])
    neg_s = summarize_1d(phys_channel_values_neg[ch])

    row = {
        "channel": ch,

        # Raw value stats
        "raw_min": raw_s["min"],
        "raw_max": raw_s["max"],
        "raw_mean": raw_s["mean"],
        "raw_std": raw_s["std"],
        "raw_p1": raw_s["p1"],
        "raw_p99": raw_s["p99"],

        # Physical value stats (all classes)
        "phys_min": phys_s["min"],
        "phys_max": phys_s["max"],
        "phys_mean": phys_s["mean"],
        "phys_std": phys_s["std"],
        "phys_p1": phys_s["p1"],
        "phys_p5": phys_s["p5"],
        "phys_p50": phys_s["p50"],
        "phys_p95": phys_s["p95"],
        "phys_p99": phys_s["p99"],

        # Class-specific stats for QC
        "phys_pos_mean": pos_s["mean"],
        "phys_pos_std": pos_s["std"],
        "phys_neg_mean": neg_s["mean"],
        "phys_neg_std": neg_s["std"],
    }
    rows.append(row)

channel_stats_df = pd.DataFrame(rows)

print("\nChannel statistics (raw + physical):")
display(channel_stats_df)


# -----------------------------
# Create Normalization Parameters
# -----------------------------

# Normalization strategy: z-score standardization
#   x_norm[c] = (x_phys[c] - train_mean[c]) / train_std[c]
#
# We also record p1/p99 for optional clipping in physical space if needed later

norm_params = {}
for _, r in channel_stats_df.iterrows():
    ch = r["channel"]
    mean = float(r["phys_mean"])
    std = float(r["phys_std"])
    p1 = float(r["phys_p1"])
    p99 = float(r["phys_p99"])

    # Protect against degenerate std (prevent division by zero)
    if not np.isfinite(std) or std < 1e-8:
        std = 1.0

    norm_params[ch] = {
        "mean": mean,
        "std": std,
        "p1": p1,
        "p99": p99,
    }

# Save normalization parameters for reproducibility
with open(TRAIN_STATS_JSON, "w") as f:
    json.dump(norm_params, f, indent=2)

print(f"✓ Saved normalization params to: {TRAIN_STATS_JSON}")

norm_params_df = pd.DataFrame([
    {"channel": ch, **vals} for ch, vals in norm_params.items()
])
print("\nFinal normalization parameters:")
display(norm_params_df)


# -----------------------------
# Normalization Function
# -----------------------------

CHANNEL_TO_INDEX = {ch: i for i, ch in enumerate(CHANNEL_ORDER)}


def normalize_from_train_stats(x_raw, norm_params, clip_optical=False, clip_zscore=None):
    """
    Normalize raw data using training set statistics.

    Pipeline:
      1. Convert raw values to physical units (reflectance, elevation, etc.)
      2. Optional: clip optical bands to [0, 1] in physical space
      3. Z-score standardization using training mean/std
      4. Optional: clip z-scores to prevent extreme outliers

    Args:
        x_raw: Raw array with shape (C, H, W)
        norm_params: Dictionary of {channel: {mean, std, p1, p99}}
        clip_optical: If True, clip optical bands to [0, 1] in physical space
        clip_zscore: If not None, clip normalized values to [-clip_zscore, +clip_zscore]

    Returns:
        Normalized array with shape (C, H, W), dtype float32
    """
    # Step 1: Convert to physical units
    x_phys = raw_to_physical(x_raw)
    x_out = np.zeros_like(x_phys, dtype=np.float32)

    for ch, i in CHANNEL_TO_INDEX.items():
        vals = x_phys[i].astype(np.float32)

        # Optional: clip optical bands in physical space
        if clip_optical and i in OPTICAL_IDX:
            vals = np.clip(vals, 0.0, 1.0)

        # Z-score normalization
        mean = norm_params[ch]["mean"]
        std = norm_params[ch]["std"]
        vals = (vals - mean) / std

        # Optional: clip extreme z-score outliers
        if clip_zscore is not None:
            vals = np.clip(vals, -clip_zscore, clip_zscore)

        x_out[i] = vals

    return x_out


print("\n✓ normalize_from_train_stats(...) is ready")


# -----------------------------
# Sanity Check Normalization
# -----------------------------

# Test normalization on a few random samples
sanity_rows = TRAIN_DF.sample(n=min(6, len(TRAIN_DF)), random_state=SEED)

sanity_summaries = []
for _, row in sanity_rows.iterrows():
    x_raw = read_sample_raw_from_row(row)
    x_norm = normalize_from_train_stats(x_raw, norm_params, clip_optical=False, clip_zscore=None)

    sanity_summaries.append({
        "sample_id": row["sample_id"],
        "source_family": row["source_family"],
        "class_label": row["class_label"],
        "raw_global_min": float(x_raw.min()),
        "raw_global_max": float(x_raw.max()),
        "norm_global_min": float(x_norm.min()),
        "norm_global_max": float(x_norm.max()),
        "norm_mean_all_channels": float(x_norm.mean()),
        "norm_std_all_channels": float(x_norm.std()),
    })

sanity_df = pd.DataFrame(sanity_summaries)
print("\nQuick normalization sanity check:")
display(sanity_df)

print("\n✓ Channel statistics and normalization complete")

## Step 7: DATASET & DATALOADER IMPLEMENTATION

In [ ]:
# ============================================================
# DATASET & DATALOADER IMPLEMENTATION
# ============================================================
# This section:
# 1. Loads train/val/test splits from saved CSVs
# 2. Implements PyTorch Dataset class with normalization and augmentation
# 3. Creates balanced sampling for training (handles class imbalance)
# 4. Sets up DataLoaders for efficient batch processing
# ============================================================

# -----------------------------
# Load Split DataFrames
# -----------------------------

train_df = pd.read_csv(amazon_train_csv).copy()
val_df = pd.read_csv(amazon_val_csv).copy()
test_df = pd.read_csv(amazon_test_csv).copy()
central_asia_eval_df = pd.read_csv(central_asia_eval_csv).copy() if os.path.exists(central_asia_eval_csv) else pd.DataFrame()

print("Train / Val / Test sizes:")
print(len(train_df), len(val_df), len(test_df))
if len(central_asia_eval_df):
    print("Central Asia eval size:", len(central_asia_eval_df))


# -----------------------------
# Central Asia Label Cleanup
# -----------------------------

# Central Asia labels may be stored as float due to NaN handling in CSV
if len(central_asia_eval_df) and "class_label" in central_asia_eval_df.columns:
    central_asia_eval_df = central_asia_eval_df.copy()
    central_asia_eval_df["class_label"] = pd.to_numeric(
        central_asia_eval_df["class_label"], errors="coerce"
    )

    # Keep only labeled samples for evaluation
    ca_labeled_mask = central_asia_eval_df["class_label"].notna()
    central_asia_eval_labeled_df = central_asia_eval_df.loc[ca_labeled_mask].copy()
    if len(central_asia_eval_labeled_df):
        central_asia_eval_labeled_df["class_label"] = central_asia_eval_labeled_df["class_label"].astype(int)
    print("Central Asia labeled rows:", len(central_asia_eval_labeled_df))
else:
    central_asia_eval_labeled_df = pd.DataFrame()


# -----------------------------
# Online Data Augmentation Functions
# -----------------------------

def random_flip_rot90(x):
    """
    Apply random horizontal/vertical flips and 90-degree rotations.

    This provides geometric invariance - archaeological features should be
    recognizable regardless of orientation.

    Args:
        x: Numpy array with shape (C, H, W)

    Returns:
        Augmented array with shape (C, H, W)
    """
    # Horizontal flip (50% probability)
    if np.random.rand() < 0.5:
        x = np.flip(x, axis=2).copy()

    # Vertical flip (50% probability)
    if np.random.rand() < 0.5:
        x = np.flip(x, axis=1).copy()

    # Random 0°/90°/180°/270° rotation
    k = np.random.randint(0, 4)
    if k > 0:
        x = np.rot90(x, k=k, axes=(1, 2)).copy()

    return x


def random_channel_noise(x, p=0.15, sigma=0.03):
    """
    Add small Gaussian noise to normalized channels.

    This is conservative - mild noise helps model generalization without
    distorting the signal too much. Applied in normalized (z-score) space.

    Args:
        x: Normalized array with shape (C, H, W)
        p: Probability of applying noise (default: 0.15)
        sigma: Standard deviation of Gaussian noise (default: 0.03)

    Returns:
        Array with optional noise added
    """
    if np.random.rand() >= p:
        return x
    noise = np.random.normal(loc=0.0, scale=sigma, size=x.shape).astype(np.float32)
    return x + noise


def apply_train_augmentation(x_norm):
    """
    Apply full training augmentation pipeline.

    Only used during training - validation/test use raw normalized data.

    Args:
        x_norm: Normalized float32 array with shape (C, H, W)

    Returns:
        Augmented array with shape (C, H, W)
    """
    x = random_flip_rot90(x_norm)
    x = random_channel_noise(x, p=0.15, sigma=0.03)
    return x


# -----------------------------
# PyTorch Dataset Class
# -----------------------------

class GeoglyphClassificationDataset(Dataset):
    """
    PyTorch Dataset for archaeological site classification.

    Handles:
    - Reading from multiple storage backends (Zarr, local folders)
    - Normalization using training statistics
    - Optional training augmentation
    - Metadata passthrough for analysis
    """

    def __init__(
        self,
        df: pd.DataFrame,
        norm_params: dict,
        training: bool = False,
        return_metadata: bool = True,
    ):
        """
        Initialize dataset.

        Args:
            df: DataFrame with sample information (paths, labels, etc.)
            norm_params: Normalization parameters from training set
            training: If True, apply data augmentation
            return_metadata: If True, return (x, y, metadata) else (x, y)
        """
        self.df = df.reset_index(drop=True).copy()
        self.norm_params = norm_params
        self.training = training
        self.return_metadata = return_metadata

    def __len__(self):
        return len(self.df)

    def _read_row_raw(self, row):
        """Read raw data for a single sample."""
        return read_sample_raw_from_row(row)

    def _normalize(self, x_raw):
        """
        Normalize raw data using training statistics.

        Conservative approach:
        - No optical clipping (keep full dynamic range)
        - No z-score clipping (preserve outliers for now)

        Args:
            x_raw: Raw array from storage

        Returns:
            Normalized array, dtype float32
        """
        x_norm = normalize_from_train_stats(
            x_raw,
            norm_params=self.norm_params,
            clip_optical=False,
            clip_zscore=None
        )
        return x_norm.astype(np.float32)

    def __getitem__(self, idx):
        """
        Get a single sample.

        Returns:
            If return_metadata=True: (x, y, metadata_dict)
            If return_metadata=False: (x, y)

            Where:
                x: Tensor with shape (C, H, W), dtype float32
                y: Tensor with shape (), dtype float32 (class label)
                metadata_dict: Dictionary with sample information
        """
        row = self.df.iloc[idx]

        # Read and normalize
        x_raw = self._read_row_raw(row)
        x = self._normalize(x_raw)

        # Apply augmentation only during training
        if self.training:
            x = apply_train_augmentation(x)

        # Convert to PyTorch tensor
        x = torch.from_numpy(x).float()

        # Get label (handle missing labels for unlabeled data)
        y = row.get("class_label", np.nan)
        if pd.isna(y):
            y = -1  # Sentinel value for unlabeled samples
        y = torch.tensor(float(y), dtype=torch.float32)

        if not self.return_metadata:
            return x, y

        # Collect metadata for analysis/debugging
        meta = {
            "sample_id": str(row.get("sample_id", "")),
            "group_id": str(row.get("group_id", "")),
            "source_family": str(row.get("source_family", "")),
            "storage_type": str(row.get("storage_type", "")),
            "region": str(row.get("region", "")),
            "site_id": None if pd.isna(row.get("site_id", None)) else str(row.get("site_id")),
            "grid_id": str(row.get("grid_id", "")),
            "local_path": None if pd.isna(row.get("local_path", None)) else str(row.get("local_path")),
            "zarr_index": None if pd.isna(row.get("zarr_index", None)) else int(row.get("zarr_index")),
        }

        return x, y, meta


# -----------------------------
# Create Dataset Instances
# -----------------------------

train_ds = GeoglyphClassificationDataset(
    train_df, norm_params=norm_params, training=True, return_metadata=True
)
val_ds = GeoglyphClassificationDataset(
    val_df, norm_params=norm_params, training=False, return_metadata=True
)
test_ds = GeoglyphClassificationDataset(
    test_df, norm_params=norm_params, training=False, return_metadata=True
)

if len(central_asia_eval_labeled_df):
    ca_eval_ds = GeoglyphClassificationDataset(
        central_asia_eval_labeled_df, norm_params=norm_params, training=False, return_metadata=True
    )
else:
    ca_eval_ds = None

print("Datasets ready:")
print("  train_ds:", len(train_ds))
print("  val_ds:  ", len(val_ds))
print("  test_ds: ", len(test_ds))
if ca_eval_ds is not None:
    print("  ca_eval_ds:", len(ca_eval_ds))


# -----------------------------
# Balanced Sampling for Training
# -----------------------------

# Handle class imbalance with weighted sampling
# This ensures the model sees a balanced distribution during training
# even if the dataset has unequal class counts

train_labels_np = train_df["class_label"].astype(int).values
class_counts = Counter(train_labels_np)
print("\nTrain class counts:", class_counts)

# Weight samples inversely proportional to class frequency
class_weights = {
    cls: 1.0 / count for cls, count in class_counts.items()
}
sample_weights = np.array([class_weights[y] for y in train_labels_np], dtype=np.float64)
sample_weights = torch.DoubleTensor(sample_weights)

# WeightedRandomSampler: samples with replacement according to weights
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("WeightedRandomSampler ready (balances class distribution)")


# -----------------------------
# Custom Collate Function
# -----------------------------

def cls_collate_fn(batch):
    """
    Collate function for batching samples with metadata.

    Args:
        batch: List of (x, y, meta) tuples from dataset

    Returns:
        xs: Batched inputs, shape (B, C, H, W)
        ys: Batched labels, shape (B,)
        metas: List of metadata dictionaries
    """
    xs, ys, metas = [], [], []
    for item in batch:
        if len(item) == 3:
            x, y, meta = item
        else:
            x, y = item
            meta = None
        xs.append(x)
        ys.append(y)
        metas.append(meta)

    xs = torch.stack(xs, dim=0)  # (B, C, H, W)
    ys = torch.stack(ys, dim=0)  # (B,)
    return xs, ys, metas


# -----------------------------
# Create DataLoaders
# -----------------------------

# Training loader: uses weighted sampler for class balance
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,  # Weighted sampling instead of shuffle
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),  # Faster GPU transfer
    collate_fn=cls_collate_fn,
    drop_last=False
)

# Validation loader: sequential, no augmentation
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Keep order for reproducible evaluation
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=cls_collate_fn,
    drop_last=False
)

# Test loader: sequential, no augmentation
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=cls_collate_fn,
    drop_last=False
)

# Central Asia loader: sequential, no augmentation
if ca_eval_ds is not None:
    ca_eval_loader = DataLoader(
        ca_eval_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=cls_collate_fn,
        drop_last=False
    )
else:
    ca_eval_loader = None

print("\nDataLoaders ready")
print("  train batches:", len(train_loader))
print("  val batches:  ", len(val_loader))
print("  test batches: ", len(test_loader))
if ca_eval_loader is not None:
    print("  ca batches:   ", len(ca_eval_loader))


# -----------------------------
# QC: Inspect One Training Batch
# -----------------------------

xb, yb, mb = next(iter(train_loader))

print("\nOne train batch:")
print("x shape:", xb.shape)
print("y shape:", yb.shape)
print("x dtype:", xb.dtype)
print("y dtype:", yb.dtype)
print("x min/max:", float(xb.min()), float(xb.max()))
print("x mean/std:", float(xb.mean()), float(xb.std()))
print("y unique/counts:", torch.unique(yb, return_counts=True))

print("\nFirst 3 metadata entries:")
for i in range(min(3, len(mb))):
    print(mb[i])


# -----------------------------
# QC: Visualize Normalized Channels
# -----------------------------

def show_sample_channels(dataset, idx=0, n_cols=4, figsize=(12, 10)):
    """
    Visualize all channels of a single sample after normalization.

    Useful for quality control:
    - Verify normalization is working correctly
    - Check for artifacts or data issues
    - Understand channel contributions

    Args:
        dataset: PyTorch Dataset instance
        idx: Sample index to visualize
        n_cols: Number of columns in grid
        figsize: Figure size (width, height)
    """
    x, y, meta = dataset[idx]
    x_np = x.numpy()

    n_ch = x_np.shape[0]
    n_rows = int(np.ceil(n_ch / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)

    for i in range(len(axes)):
        ax = axes[i]
        ax.axis("off")
        if i < n_ch:
            im = ax.imshow(x_np[i], cmap="gray")
            ax.set_title(CHANNEL_ORDER[i])
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(
        f"sample_id={meta['sample_id']} | y={float(y.item()):.0f} | source={meta['source_family']}",
        y=1.02
    )
    plt.tight_layout()
    plt.show()


# Visualize first training sample
show_sample_channels(train_ds, idx=0)

print("\n✓ Dataset and DataLoader setup complete")

## Step 8: MODEL ARCHITECTURE

In [ ]:
# ============================================================
# MODEL ARCHITECTURE
# ============================================================
# Classification model for archaeological site detection.
#
# Architecture: CNN-based binary classifier
# - Convolutional feature extraction with progressive downsampling
# - Dual pooling (avg + max) for robust global features
# - Fully connected classifier head
#
# Design principles:
# - Lightweight (few parameters for efficient training)
# - Spatial invariance through pooling
# - Regularization via dropout and batch normalization
# ============================================================

# -----------------------------
# Building Blocks
# -----------------------------

class ConvBNReLU(nn.Module):
    """
    Standard convolutional block: Conv2d -> BatchNorm -> ReLU

    This is the fundamental building block used throughout the network.
    BatchNorm provides normalization and slight regularization.
    """

    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        """
        Args:
            in_ch: Number of input channels
            out_ch: Number of output channels
            k: Kernel size (default: 3x3)
            s: Stride (default: 1)
            p: Padding (default: 1, maintains spatial dimensions)
        """
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# -----------------------------
# Main Classifier Network
# -----------------------------

class SimpleGeoClassifier(nn.Module):
    """
    CNN-based binary classifier for archaeological site detection.

    Architecture:
        Input: (B, 11, 100, 100) - 11 satellite/terrain channels

        Feature Extraction:
        - Block 1: 11 -> 24 channels, spatial 100x100 -> 50x50
        - Block 2: 24 -> 48 channels, spatial 50x50 -> 25x25
        - Block 3: 48 -> 96 channels, spatial 25x25 -> 12x12
        - Block 4: 96 -> 144 channels, spatial 12x12 (no pooling)

        Global Pooling:
        - AdaptiveAvgPool: captures average response
        - AdaptiveMaxPool: captures peak response
        - Concatenate both for richer feature representation

        Classifier Head:
        - 288 -> 128 -> 32 -> 1
        - Dropout for regularization

        Output: Single logit (use sigmoid for probability)
    """

    def __init__(self, in_ch=11, base_ch=24, dropout=0.20):
        """
        Args:
            in_ch: Number of input channels (default: 11 for this dataset)
            base_ch: Base number of channels (scales up in deeper layers)
            dropout: Dropout probability for classifier head
        """
        super().__init__()

        # Convolutional feature extractor
        # Progressive channel expansion: 11 -> 24 -> 48 -> 96 -> 144
        # Progressive spatial reduction: 100 -> 50 -> 25 -> 12
        self.features = nn.Sequential(
            # Block 1: Initial feature extraction (100x100 -> 50x50)
            ConvBNReLU(in_ch, base_ch),              # 11 -> 24 channels
            ConvBNReLU(base_ch, base_ch),            # 24 -> 24 channels
            nn.MaxPool2d(2),                         # 100x100 -> 50x50

            # Block 2: Intermediate features (50x50 -> 25x25)
            ConvBNReLU(base_ch, base_ch * 2),        # 24 -> 48 channels
            ConvBNReLU(base_ch * 2, base_ch * 2),    # 48 -> 48 channels
            nn.MaxPool2d(2),                         # 50x50 -> 25x25

            # Block 3: Deep features (25x25 -> 12x12)
            ConvBNReLU(base_ch * 2, base_ch * 4),    # 48 -> 96 channels
            ConvBNReLU(base_ch * 4, base_ch * 4),    # 96 -> 96 channels
            nn.MaxPool2d(2),                         # 25x25 -> 12x12

            # Block 4: High-level features (12x12, no pooling)
            ConvBNReLU(base_ch * 4, base_ch * 6),    # 96 -> 144 channels
            ConvBNReLU(base_ch * 6, base_ch * 6),    # 144 -> 144 channels
        )

        # Global pooling: collapse spatial dimensions to single values
        # Using both avg and max captures different aspects of feature maps
        self.avgpool = nn.AdaptiveAvgPool2d(1)  # (B, C, H, W) -> (B, C, 1, 1)
        self.maxpool = nn.AdaptiveMaxPool2d(1)  # (B, C, H, W) -> (B, C, 1, 1)

        # Feature dimension after concatenating avg and max pooling
        feat_dim = base_ch * 6 * 2  # 144 * 2 = 288

        # Classification head
        # Progressive dimensionality reduction with dropout regularization
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 128),           # 288 -> 128
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),                # Dropout (default: 0.20)

            nn.Linear(128, 32),                 # 128 -> 32
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),          # Reduced dropout (0.10)

            nn.Linear(32, 1),                   # 32 -> 1 (logit)
        )

        # Initialize weights using best practices
        self.apply(self._init_weights)

    def _init_weights(self, m):
        """
        Initialize network weights using Kaiming initialization.

        Kaiming initialization is designed for ReLU activations and helps
        prevent vanishing/exploding gradients during training.

        Args:
            m: Module to initialize
        """
        if isinstance(m, nn.Conv2d):
            # Kaiming normal for conv layers
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.0)

        elif isinstance(m, nn.BatchNorm2d):
            # Standard initialization for batch norm
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0.0)

        elif isinstance(m, nn.Linear):
            # Kaiming normal for linear layers
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: Input tensor, shape (B, C, H, W)
               B = batch size
               C = 11 channels
               H = W = 100 pixels

        Returns:
            logit: Output tensor, shape (B,)
                   Raw logits (apply sigmoid for probabilities)
        """
        # Extract convolutional features
        x = self.features(x)  # (B, 144, 12, 12)

        # Global pooling and concatenation
        a = self.avgpool(x).flatten(1)  # (B, 144)
        m = self.maxpool(x).flatten(1)  # (B, 144)
        feat = torch.cat([a, m], dim=1) # (B, 288)

        # Classification
        logit = self.classifier(feat).squeeze(1)  # (B,)

        return logit


# -----------------------------
# Utility Functions
# -----------------------------

def count_parameters(model):
    """
    Count total and trainable parameters in a model.

    Args:
        model: PyTorch model

    Returns:
        total: Total number of parameters
        trainable: Number of trainable parameters
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# -----------------------------
# Build and Initialize Model
# -----------------------------

model = SimpleGeoClassifier(
    in_ch=N_CHANNELS,    # 11 input channels
    base_ch=24,          # Base channel width
    dropout=0.20         # 20% dropout in classifier head
).to(device)

n_total, n_trainable = count_parameters(model)

print(model)
print(f"\nTotal params:     {n_total:,}")
print(f"Trainable params: {n_trainable:,}")


# -----------------------------
# Sanity Check: Forward Pass
# -----------------------------

# Verify model produces expected output shapes and value ranges
model.eval()
with torch.no_grad():
    xb, yb, mb = next(iter(train_loader))
    xb = xb.to(device)
    logits = model(xb)
    probs = torch.sigmoid(logits)

print("\nForward pass sanity check:")
print("input shape:  ", xb.shape)          # Expected: (B, 11, 100, 100)
print("logits shape: ", logits.shape)      # Expected: (B,)
print("logits dtype: ", logits.dtype)      # Expected: torch.float32
print("logits min/max:", float(logits.min().cpu()), float(logits.max().cpu()))
print("probs min/max: ", float(probs.min().cpu()), float(probs.max().cpu()))  # Should be in [0, 1]
print("probs mean:    ", float(probs.mean().cpu()))  # Should be reasonable (~0.5 for balanced)

print("\n✓ Model architecture ready")

In [ ]:
# # ============================================================
# # LOAD BEST CHECKPOINT
# # ============================================================
# # This section loads the best model checkpoint from training.
# #
# # Checkpoint strategy:
# # - Checkpoints are saved during training to GCS
# # - If not found locally, automatically download from GCS
# # - Load model weights from the best validation performance epoch
# #
# # This allows:
# # - Resuming evaluation without retraining
# # - Sharing trained models across environments
# # - Reproducible evaluation results
# # ============================================================

# # -----------------------------
# # Define Checkpoint Paths
# # -----------------------------

# # Local path where checkpoint should be stored
# BEST_CKPT_PATH = os.path.join(LOCAL_CKPT, BEST_CKPT_NAME)

# # GCS path where checkpoint is stored (backup/sharing location)
# GCS_BEST_CKPT_PATH = os.path.join(
#     GCS_ZARR_ROOT,
#     f"stage1_runs/{STAGE1_RUN_NAME}/",
#     BEST_CKPT_NAME
# )


# # -----------------------------
# # Download Checkpoint if Needed
# # -----------------------------

# # Check if checkpoint exists locally
# if not os.path.exists(BEST_CKPT_PATH):
#     print(f"Best checkpoint not found locally at {BEST_CKPT_PATH}.")
#     print("Attempting to download from GCS...")

#     try:
#         # Ensure the local checkpoint directory exists
#         os.makedirs(LOCAL_CKPT, exist_ok=True)

#         # Download from GCS
#         download_gcs_file_or_dir(GCS_BEST_CKPT_PATH, BEST_CKPT_PATH)
#         print(f"✓ Successfully downloaded {BEST_CKPT_NAME} from GCS")

#     except Exception as e:
#         print(f"ERROR: Failed to download checkpoint from GCS: {e}")
#         print("Please ensure:")
#         print("  1. The checkpoint exists at the GCS path")
#         print("  2. Your Google Cloud authentication is valid")
#         print("  3. You have read permissions for the GCS bucket")
#         raise  # Re-raise exception to stop execution


# # -----------------------------
# # Load Checkpoint into Model
# # -----------------------------

# if os.path.exists(BEST_CKPT_PATH):
#     # Load checkpoint file
#     # map_location ensures compatibility across CPU/GPU environments
#     best_payload = torch.load(BEST_CKPT_PATH, map_location=device)

#     # Restore model weights
#     model.load_state_dict(best_payload["model_state_dict"])

#     # Set to evaluation mode (disables dropout, batch norm in eval mode)
#     model.eval()

#     # Display checkpoint information
#     print(f"✓ Loaded best checkpoint from: {BEST_CKPT_PATH}")
#     print(f"  Best epoch: {best_payload.get('epoch')}")
#     print(f"  Best validation metric: {best_payload.get('best_metric')}")

# else:
#     # This should not happen after the download attempt above
#     print(f"ERROR: Best checkpoint still not found at {BEST_CKPT_PATH}")
#     print("Please verify:")
#     print(f"  - GCS path: {GCS_BEST_CKPT_PATH}")
#     print("  - Local path: {BEST_CKPT_PATH}")
#     print("  - File permissions and bucket access")
#     raise FileNotFoundError(f"Best checkpoint not found: {BEST_CKPT_PATH}")


# # -----------------------------
# # Ensure Model is on Correct Device
# # -----------------------------

# # Move model to GPU/CPU as configured
# model.to(device)

# print(f"✓ Model ready on device: {device}")

## Step 9: TRAINING LOOP

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
# This section implements the complete training pipeline:
# 1. Loss function with class weighting (handles imbalance)
# 2. Optimizer and learning rate scheduling
# 3. Comprehensive evaluation metrics (AUC, AP, precision, recall, F-scores)
# 4. Training and validation loops with automatic mixed precision
# 5. Checkpoint saving (best and last)
# 6. Training history tracking
# ============================================================

# -----------------------------
# Loss, Optimizer, Scheduler, AMP
# -----------------------------

# Class weighting: emphasize positive class (archaeological sites)
# POS_WEIGHT > 1 reduces false negatives and improves recall
# This is critical for archaeological detection where missing a site is costly
POS_WEIGHT = 2.0
pos_weight_tensor = torch.tensor([POS_WEIGHT], device=device)

# Binary cross-entropy with logits (numerically stable)
# Applies pos_weight to penalize false negatives more heavily
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# AdamW optimizer (Adam with decoupled weight decay)
# Better generalization than standard Adam
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Reduce learning rate when validation metric plateaus
# Monitors precision at recall ≥ 0.85 (recall-focused metric)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",          # Maximize the metric
    factor=0.5,          # Reduce LR by half
    patience=2,          # Wait 2 epochs before reducing
)

# Automatic Mixed Precision (AMP) for faster training on GPU
amp_enabled = USE_AMP and torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

print("Loss:", criterion)
print("POS_WEIGHT:", POS_WEIGHT)
print("Optimizer:", optimizer.__class__.__name__)
print("Scheduler:", scheduler.__class__.__name__)
print("AMP enabled:", amp_enabled)


# -----------------------------
# Metric Helper Functions
# -----------------------------

def safe_roc_auc(y_true, y_prob):
    """
    Compute ROC-AUC score safely (handles edge cases).

    Returns NaN if only one class is present in y_true.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_ap(y_true, y_prob):
    """
    Compute Average Precision (AP) score safely.

    Returns NaN if only one class is present in y_true.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def fbeta_from_pr(precision, recall, beta=1.0):
    """
    Compute F-beta score from precision and recall.

    F-beta score weights recall β times as important as precision.
    - F1 (β=1): Balanced precision and recall
    - F2 (β=2): Emphasizes recall over precision (better for site detection)

    Args:
        precision: Precision value
        recall: Recall value
        beta: Weight of recall vs precision

    Returns:
        F-beta score
    """
    denom = (beta**2 * precision + recall)
    if denom <= 0:
        return 0.0
    return (1 + beta**2) * precision * recall / denom


def compute_threshold_metrics(y_true, y_prob, threshold=0.5):
    """
    Compute classification metrics at a specific probability threshold.

    Args:
        y_true: True binary labels
        y_prob: Predicted probabilities
        threshold: Classification threshold (default: 0.5)

    Returns:
        Dictionary with TP, TN, FP, FN, precision, recall, specificity,
        accuracy, F1, and F2 scores
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    # Confusion matrix components
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    # Derived metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    accuracy = (tp + tn) / max(1, len(y_true))
    f1 = fbeta_from_pr(precision, recall, beta=1.0)
    f2 = fbeta_from_pr(precision, recall, beta=2.0)

    return {
        "threshold": float(threshold),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "accuracy": float(accuracy),
        "f1": float(f1),
        "f2": float(f2),
    }


def find_best_fbeta_threshold(y_true, y_prob, beta=1.0):
    """
    Find threshold that maximizes F-beta score.

    Searches across all thresholds from precision-recall curve.

    Args:
        y_true: True binary labels
        y_prob: Predicted probabilities
        beta: F-beta parameter (1.0 for F1, 2.0 for F2)

    Returns:
        best_threshold: Optimal threshold
        best_stats: Metrics dict at that threshold
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    if len(thresholds) == 0:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
        return 0.5, stats

    best_thr = 0.5
    best_score = -1.0
    best_stats = None

    for thr in thresholds:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=float(thr))
        score = stats["f2"] if beta == 2.0 else stats["f1"] if beta == 1.0 else fbeta_from_pr(
            stats["precision"], stats["recall"], beta=beta
        )
        if score > best_score:
            best_score = score
            best_thr = float(thr)
            best_stats = stats.copy()
            best_stats[f"f{beta}".replace(".0", "")] = float(score)

    return best_thr, best_stats


def find_best_f1_threshold(y_true, y_prob):
    """Find threshold that maximizes F1 score."""
    return find_best_fbeta_threshold(y_true, y_prob, beta=1.0)


def find_best_f2_threshold(y_true, y_prob):
    """Find threshold that maximizes F2 score."""
    return find_best_fbeta_threshold(y_true, y_prob, beta=2.0)


def find_best_precision_under_recall(y_true, y_prob, min_recall=0.85):
    """
    Find threshold that maximizes precision while maintaining minimum recall.

    This is critical for archaeological site detection:
    - We want high recall (don't miss sites)
    - Among thresholds achieving min_recall, pick the one with best precision

    Args:
        y_true: True binary labels
        y_prob: Predicted probabilities
        min_recall: Minimum acceptable recall (default: 0.85)

    Returns:
        best_threshold: Optimal threshold
        best_stats: Metrics dict at that threshold
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    if len(thresholds) == 0:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
        return 0.5, stats

    best_thr = None
    best_stats = None
    best_precision = -1.0

    for thr in thresholds:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=float(thr))
        if stats["recall"] >= min_recall and stats["precision"] > best_precision:
            best_precision = stats["precision"]
            best_thr = float(thr)
            best_stats = stats.copy()

    # Fallback if no threshold meets recall requirement
    if best_thr is None:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
        return 0.5, stats

    return best_thr, best_stats


# -----------------------------
# Training and Evaluation Functions
# -----------------------------

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    """
    Train model for one epoch.

    Args:
        model: PyTorch model
        loader: Training DataLoader
        optimizer: Optimizer
        criterion: Loss function
        scaler: GradScaler for AMP
        device: Device (CPU/GPU)

    Returns:
        Dictionary with epoch metrics (loss, AUC, AP, threshold metrics)
    """
    model.train()

    running_loss = 0.0
    all_y = []
    all_prob = []
    all_logit = []

    pbar = tqdm(loader, desc="Train", leave=False)

    for xb, yb, _ in pbar:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Zero gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with AMP
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            logits = model(xb)
            loss = criterion(logits, yb)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Compute probabilities for metrics
        with torch.no_grad():
            probs = torch.sigmoid(logits)

        # Accumulate for epoch-level metrics
        running_loss += loss.item() * xb.size(0)
        all_y.append(yb.detach().cpu().numpy())
        all_prob.append(probs.detach().cpu().numpy())
        all_logit.append(logits.detach().cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # Compute epoch-level metrics
    y_true = np.concatenate(all_y)
    y_prob = np.concatenate(all_prob)
    y_logit = np.concatenate(all_logit)

    epoch_loss = running_loss / len(loader.dataset)
    auc = safe_roc_auc(y_true, y_prob)
    ap = safe_ap(y_true, y_prob)
    thr05 = compute_threshold_metrics(y_true, y_prob, threshold=0.5)

    return {
        "loss": float(epoch_loss),
        "auc": float(auc) if np.isfinite(auc) else np.nan,
        "ap": float(ap) if np.isfinite(ap) else np.nan,
        "thr05_precision": thr05["precision"],
        "thr05_recall": thr05["recall"],
        "thr05_f1": thr05["f1"],
        "thr05_f2": thr05["f2"],
        "y_true": y_true,
        "y_prob": y_prob,
        "y_logit": y_logit,
    }


@torch.no_grad()
def evaluate_classifier(model, loader, criterion, device, desc="Eval"):
    """
    Evaluate model on validation/test set.

    Computes comprehensive metrics:
    - Loss, AUC, AP (summary metrics)
    - Metrics at threshold 0.5
    - Best F1 threshold and metrics
    - Best F2 threshold and metrics
    - Best precision at recall ≥ 0.85

    Args:
        model: PyTorch model
        loader: Evaluation DataLoader
        criterion: Loss function
        device: Device (CPU/GPU)
        desc: Description for progress bar

    Returns:
        Dictionary with comprehensive metrics and predictions DataFrame
    """
    model.eval()

    running_loss = 0.0
    all_y = []
    all_prob = []
    all_logit = []
    all_meta = []

    pbar = tqdm(loader, desc=desc, leave=False)

    for xb, yb, metas in pbar:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Forward pass with AMP
        with torch.amp.autocast("cuda", enabled=(USE_AMP and torch.cuda.is_available())):
            logits = model(xb)
            loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)

        # Accumulate results
        running_loss += loss.item() * xb.size(0)
        all_y.append(yb.detach().cpu().numpy())
        all_prob.append(probs.detach().cpu().numpy())
        all_logit.append(logits.detach().cpu().numpy())
        all_meta.extend(metas)

    # Concatenate all predictions
    y_true_all = np.concatenate(all_y).astype(float)
    y_prob_all = np.concatenate(all_prob).astype(float)
    y_logit_all = np.concatenate(all_logit).astype(float)

    # Create predictions DataFrame with metadata
    pred_df = pd.DataFrame({
        "sample_id": [m.get("sample_id") if m is not None else None for m in all_meta],
        "group_id": [m.get("group_id") if m is not None else None for m in all_meta],
        "source_family": [m.get("source_family") if m is not None else None for m in all_meta],
        "region": [m.get("region") if m is not None else None for m in all_meta],
        "site_id": [m.get("site_id") if m is not None else None for m in all_meta],
        "grid_id": [m.get("grid_id") if m is not None else None for m in all_meta],
        "y_true": y_true_all,
        "y_logit": y_logit_all,
        "y_prob": y_prob_all,
    })

    # Filter to valid binary labels with finite predictions
    valid_mask = (
        pred_df["y_true"].isin([0.0, 1.0]) &
        np.isfinite(pred_df["y_logit"].values) &
        np.isfinite(pred_df["y_prob"].values)
    )

    dropped = int((~valid_mask).sum())
    if dropped > 0:
        print(f"[{desc}] Dropping {dropped} invalid rows before metrics")

    pred_df = pred_df.loc[valid_mask].reset_index(drop=True)

    # Extract cleaned arrays for metric computation
    y_true = pred_df["y_true"].values.astype(int)
    y_prob = pred_df["y_prob"].values.astype(float)
    y_logit = pred_df["y_logit"].values.astype(float)

    # Compute comprehensive metrics
    epoch_loss = running_loss / len(loader.dataset)
    auc = safe_roc_auc(y_true, y_prob)
    ap = safe_ap(y_true, y_prob)

    thr05 = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
    best_f1_thr, best_f1_stats = find_best_f1_threshold(y_true, y_prob)
    best_f2_thr, best_f2_stats = find_best_f2_threshold(y_true, y_prob)
    best_prec_thr, best_prec_stats = find_best_precision_under_recall(y_true, y_prob, min_recall=0.85)

    return {
        "loss": float(epoch_loss),
        "auc": float(auc) if np.isfinite(auc) else np.nan,
        "ap": float(ap) if np.isfinite(ap) else np.nan,
        "thr05": thr05,
        "best_f1_thr": float(best_f1_thr),
        "best_f1_stats": best_f1_stats,
        "best_f2_thr": float(best_f2_thr),
        "best_f2_stats": best_f2_stats,
        "best_prec_thr_recall85": float(best_prec_thr),
        "best_prec_stats_recall85": best_prec_stats,
        "y_true": y_true,
        "y_prob": y_prob,
        "y_logit": y_logit,
        "pred_df": pred_df,
    }


# -----------------------------
# Checkpoint Management
# -----------------------------

BEST_CKPT_PATH = os.path.join(LOCAL_CKPT, BEST_CKPT_NAME)
LAST_CKPT_PATH = os.path.join(LOCAL_CKPT, LAST_CKPT_NAME)
HISTORY_JSON_PATH = os.path.join(LOCAL_REPORTS, f"{STAGE1_RUN_NAME}_history.json")


def save_checkpoint(path, epoch, model, optimizer, scheduler, best_metric, extra=None):
    """
    Save training checkpoint with model weights and training state.

    Args:
        path: Checkpoint file path
        epoch: Current epoch number
        model: PyTorch model
        optimizer: Optimizer
        scheduler: Learning rate scheduler
        best_metric: Best validation metric achieved so far
        extra: Additional metadata to save
    """
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": best_metric,
        "config": {
            "run_name": STAGE1_RUN_NAME,
            "n_channels": N_CHANNELS,
            "image_size": IMAGE_SIZE,
            "lr": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "batch_size": BATCH_SIZE,
            "seed": SEED,
            "epochs": NUM_EPOCHS,
            "pos_weight": POS_WEIGHT,
        },
        "extra": extra if extra is not None else {},
    }
    torch.save(payload, path)


# -----------------------------
# Main Training Loop
# -----------------------------

history = []
best_val_metric = -np.inf
best_epoch = -1
best_metric_name = "val_best_prec_recall85"  # Optimize for precision at recall ≥ 0.85

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 70)

    # Training phase
    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=device
    )

    # Validation phase
    val_metrics = evaluate_classifier(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        desc="Val"
    )

    # Get current validation metric for tracking best model
    current_val_metric = val_metrics["best_prec_stats_recall85"]["precision"]

    # Update learning rate based on validation metric
    scheduler.step(current_val_metric if np.isfinite(current_val_metric) else -1e9)

    current_lr = optimizer.param_groups[0]["lr"]

    # Record epoch history
    epoch_row = {
        "epoch": epoch,
        "lr": current_lr,

        # Training metrics
        "train_loss": train_metrics["loss"],
        "train_auc": train_metrics["auc"],
        "train_ap": train_metrics["ap"],
        "train_thr05_precision": train_metrics["thr05_precision"],
        "train_thr05_recall": train_metrics["thr05_recall"],
        "train_thr05_f1": train_metrics["thr05_f1"],
        "train_thr05_f2": train_metrics["thr05_f2"],

        # Validation metrics
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["auc"],
        "val_ap": val_metrics["ap"],
        "val_thr05_precision": val_metrics["thr05"]["precision"],
        "val_thr05_recall": val_metrics["thr05"]["recall"],
        "val_thr05_f1": val_metrics["thr05"]["f1"],
        "val_thr05_f2": val_metrics["thr05"]["f2"],

        # Best F1 metrics
        "val_best_f1_thr": val_metrics["best_f1_thr"],
        "val_best_f1": val_metrics["best_f1_stats"]["f1"],

        # Best F2 metrics
        "val_best_f2_thr": val_metrics["best_f2_thr"],
        "val_best_f2": val_metrics["best_f2_stats"]["f2"],

        # Best precision at recall ≥ 0.85 (primary metric)
        "val_best_prec_thr_recall85": val_metrics["best_prec_thr_recall85"],
        "val_best_prec_recall85": val_metrics["best_prec_stats_recall85"]["precision"],
        "val_best_prec_recall85_recall": val_metrics["best_prec_stats_recall85"]["recall"],
        "val_best_prec_recall85_f1": val_metrics["best_prec_stats_recall85"]["f1"],
        "val_best_prec_recall85_f2": val_metrics["best_prec_stats_recall85"]["f2"],
    }
    history.append(epoch_row)

    # Print epoch summary
    print(
        f"train loss={train_metrics['loss']:.4f} | "
        f"train AP={train_metrics['ap']:.4f} | "
        f"train AUC={train_metrics['auc']:.4f}"
    )
    print(
        f"val   loss={val_metrics['loss']:.4f} | "
        f"val   AP={val_metrics['ap']:.4f} | "
        f"val   AUC={val_metrics['auc']:.4f}"
    )
    print(
        f"val @0.5: precision={val_metrics['thr05']['precision']:.4f}, "
        f"recall={val_metrics['thr05']['recall']:.4f}, "
        f"f1={val_metrics['thr05']['f1']:.4f}, "
        f"f2={val_metrics['thr05']['f2']:.4f}"
    )
    print(
        f"val best-F1 threshold={val_metrics['best_f1_thr']:.4f}, "
        f"F1={val_metrics['best_f1_stats']['f1']:.4f}"
    )
    print(
        f"val best-F2 threshold={val_metrics['best_f2_thr']:.4f}, "
        f"F2={val_metrics['best_f2_stats']['f2']:.4f}"
    )
    print(
        f"val best-precision@recall>=0.85 threshold={val_metrics['best_prec_thr_recall85']:.4f}, "
        f"precision={val_metrics['best_prec_stats_recall85']['precision']:.4f}, "
        f"recall={val_metrics['best_prec_stats_recall85']['recall']:.4f}, "
        f"f2={val_metrics['best_prec_stats_recall85']['f2']:.4f}"
    )
    print(f"lr={current_lr:.2e}")

    # Save last checkpoint (always)
    save_checkpoint(
        LAST_CKPT_PATH,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        best_metric=best_val_metric,
        extra={
            "history_row": epoch_row,
            "best_metric_name": best_metric_name,
        }
    )

    # Save best checkpoint (when validation metric improves)
    if np.isfinite(current_val_metric) and current_val_metric > best_val_metric:
        best_val_metric = current_val_metric
        best_epoch = epoch

        save_checkpoint(
            BEST_CKPT_PATH,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_metric=best_val_metric,
            extra={
                "history_row": epoch_row,
                "best_metric_name": best_metric_name,
                "val_best_f1_thr": val_metrics["best_f1_thr"],
                "val_best_f2_thr": val_metrics["best_f2_thr"],
                "val_best_prec_thr_recall85": val_metrics["best_prec_thr_recall85"],
            }
        )
        print(f"✓ Saved new best checkpoint -> {BEST_CKPT_PATH}")

    # Save training history (updated after each epoch)
    with open(HISTORY_JSON_PATH, "w") as f:
        json.dump(history, f, indent=2)

# Training complete
print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Best {best_metric_name}: {best_val_metric:.4f} at epoch {best_epoch}")
print(f"Best checkpoint: {BEST_CKPT_PATH}")
print(f"Last checkpoint: {LAST_CKPT_PATH}")
print(f"History JSON: {HISTORY_JSON_PATH}")
print("=" * 70)

In [ ]:
# ============================================================
# SAVE BEST CHECKPOINT TO GCS
# ============================================================
# Upload the best model checkpoint to Google Cloud Storage
# for backup, sharing, and deployment
# ============================================================

GCS_STAGE1_RUN_DIR = f"{GCS_ZARR_ROOT}stage1_runs/{STAGE1_RUN_NAME}/"
print("Saving best checkpoint to:", GCS_STAGE1_RUN_DIR)

!gsutil -m cp "{BEST_CKPT_PATH}" "{GCS_STAGE1_RUN_DIR}"

print("✓ Saved:", BEST_CKPT_PATH)
print("✓ GCS dir:", GCS_STAGE1_RUN_DIR)

## Step 10: EVALUATION

In [ ]:
# ============================================================
# CLEAN CENTRAL ASIA LABELS & REBUILD EVAL LOADER
# ============================================================
# The Central Asia dataset may have inconsistent label formatting
# (floats, NaNs, non-binary values) due to CSV storage and external
# data sources. This section:
# 1. Cleans and validates labels to binary {0, 1}
# 2. Filters out invalid/unlabeled samples
# 3. Rebuilds the evaluation dataset and DataLoader
# 4. Performs quality checks on the cleaned data
# ============================================================

# -----------------------------
# Clean Central Asia Labels
# -----------------------------

print("Before cleaning Central Asia labels:")
print(central_asia_eval_labeled_df["class_label"].value_counts(dropna=False).sort_index())

# Create copy to avoid modifying original
central_asia_eval_labeled_df = central_asia_eval_labeled_df.copy()

# Convert to numeric (handles string labels, forces NaN for invalid values)
central_asia_eval_labeled_df["class_label"] = pd.to_numeric(
    central_asia_eval_labeled_df["class_label"], errors="coerce"
)

# Keep only valid binary labels (0 or 1)
# This filters out NaN, non-integer values, and any other invalid labels
central_asia_eval_labeled_df = central_asia_eval_labeled_df[
    central_asia_eval_labeled_df["class_label"].isin([0, 1])
].copy()

# Convert to integer type for consistency
central_asia_eval_labeled_df["class_label"] = central_asia_eval_labeled_df["class_label"].astype(int)

print("\nAfter cleaning Central Asia labels:")
print(central_asia_eval_labeled_df["class_label"].value_counts(dropna=False).sort_index())
print("Rows kept:", len(central_asia_eval_labeled_df))


# -----------------------------
# Display Class Balance
# -----------------------------

ca_pos = int((central_asia_eval_labeled_df["class_label"] == 1).sum())
ca_neg = int((central_asia_eval_labeled_df["class_label"] == 0).sum())
ca_total = len(central_asia_eval_labeled_df)

print("\nCentral Asia class balance:")
print(f"  negatives: {ca_neg}")
print(f"  positives: {ca_pos}")
print(f"  positive ratio: {ca_pos / max(1, ca_total):.4f}")


# -----------------------------
# Rebuild Dataset & DataLoader
# -----------------------------

# Create new dataset with cleaned labels
ca_eval_ds = GeoglyphClassificationDataset(
    central_asia_eval_labeled_df,
    norm_params=norm_params,
    training=False,           # No augmentation for evaluation
    return_metadata=True
)

# Create new DataLoader
ca_eval_loader = DataLoader(
    ca_eval_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,            # Keep original order for reproducibility
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=cls_collate_fn,
    drop_last=False          # Include all samples
)

print("\nRebuilt Central Asia eval loader:")
print("  samples:", len(ca_eval_ds))
print("  batches:", len(ca_eval_loader))


# -----------------------------
# QC: Batch Sanity Check
# -----------------------------

# Verify the loader produces valid batches
xb, yb, mb = next(iter(ca_eval_loader))
print("\nOne Central Asia batch:")
print("x shape:", xb.shape)
print("y shape:", yb.shape)
print("y dtype:", yb.dtype)
print("y unique/counts:", torch.unique(yb, return_counts=True))
print("First metadata:", mb[0])


# -----------------------------
# QC: DataFrame Validation
# -----------------------------

# Check for missing values in key columns
print("\nMissing values in key columns:")
cols_to_check = [
    c for c in ["sample_id", "group_id", "region", "site_id", "grid_id", "class_label"]
    if c in central_asia_eval_labeled_df.columns
]
print(central_asia_eval_labeled_df[cols_to_check].isna().sum())

# Verify all samples are from Central Asia region
if "region" in central_asia_eval_labeled_df.columns:
    print("\nRegion counts:")
    print(central_asia_eval_labeled_df["region"].value_counts(dropna=False))

print("\n✓ Central Asia dataset cleaned and ready for evaluation")

In [ ]:
# ============================================================
# FINAL EVALUATION & EXPORT
# ============================================================
# This section performs comprehensive evaluation of the trained model:
# 1. Loads the best checkpoint from training
# 2. Evaluates on validation, test, and Central Asia datasets
# 3. Reports metrics at multiple operating thresholds (F1, F2, precision@recall≥0.85)
# 4. Exports predictions, metrics, and summary reports
#
# IMPORTANT: Thresholds are chosen from VALIDATION set only to avoid
# test set leakage. These same thresholds are then applied to test and
# Central Asia to measure generalization.
# ============================================================

# -----------------------------
# Load Best Checkpoint
# -----------------------------

best_payload = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(best_payload["model_state_dict"])
model.eval()

print("Loaded best checkpoint from:")
print(BEST_CKPT_PATH)
print("Best epoch:", best_payload.get("epoch"))
print("Best saved metric:", best_payload.get("best_metric"))

# Extract saved metadata about thresholds from training
saved_extra = best_payload.get("extra", {})
saved_best_metric_name = saved_extra.get("best_metric_name", "unknown")
saved_val_best_f1_thr = saved_extra.get("val_best_f1_thr", None)
saved_val_best_f2_thr = saved_extra.get("val_best_f2_thr", None)
saved_val_best_prec_thr_recall85 = saved_extra.get("val_best_prec_thr_recall85", None)

print("Saved best metric name:", saved_best_metric_name)
print("Saved val best-F1 threshold:", saved_val_best_f1_thr)
print("Saved val best-F2 threshold:", saved_val_best_f2_thr)
print("Saved val best-precision@recall>=0.85 threshold:", saved_val_best_prec_thr_recall85)


# -----------------------------
# Evaluate on All Splits
# -----------------------------

# Validation set (used to select thresholds)
val_results = evaluate_classifier(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    desc="Final Val"
)

# Test set (held-out for final performance assessment)
test_results = evaluate_classifier(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    desc="Final Test"
)

# Central Asia (external transfer learning evaluation)
if ca_eval_loader is not None:
    ca_results = evaluate_classifier(
        model=model,
        loader=ca_eval_loader,
        criterion=criterion,
        device=device,
        desc="Central Asia"
    )
else:
    ca_results = None


# -----------------------------
# Choose Operating Thresholds from Validation Set
# -----------------------------

# CRITICAL: Select thresholds based on VALIDATION set only
# Applying these to test/CA measures true generalization
val_thr_f1 = val_results["best_f1_thr"]
val_thr_f2 = val_results["best_f2_thr"]
val_thr_prec85 = val_results["best_prec_thr_recall85"]

print("\nChosen thresholds from VAL:")
print(f"  best-F1 threshold: {val_thr_f1:.6f}")
print(f"  best-F2 threshold: {val_thr_f2:.6f}")
print(f"  best-precision@recall>=0.85 threshold: {val_thr_prec85:.6f}")


# -----------------------------
# Helper: Summarize Evaluation Results
# -----------------------------

def summarize_eval_results(name, results, threshold_for_reporting):
    """
    Summarize evaluation results at a specific threshold.

    Args:
        name: Dataset name (e.g., 'val', 'test', 'central_asia')
        results: Output from evaluate_classifier()
        threshold_for_reporting: Operating threshold to report metrics at

    Returns:
        Dictionary with comprehensive metrics at the specified threshold
    """
    base = compute_threshold_metrics(
        results["y_true"],
        results["y_prob"],
        threshold=threshold_for_reporting
    )

    summary = {
        "name": name,
        "loss": float(results["loss"]),
        "auc": float(results["auc"]) if np.isfinite(results["auc"]) else np.nan,
        "ap": float(results["ap"]) if np.isfinite(results["ap"]) else np.nan,
        "report_threshold": float(threshold_for_reporting),

        # Metrics at specified threshold
        "precision": float(base["precision"]),
        "recall": float(base["recall"]),
        "specificity": float(base["specificity"]),
        "accuracy": float(base["accuracy"]),
        "f1": float(base["f1"]),
        "f2": float(base["f2"]),
        "tp": int(base["tp"]),
        "tn": int(base["tn"]),
        "fp": int(base["fp"]),
        "fn": int(base["fn"]),

        # Internal best thresholds (for reference/comparison)
        "best_f1_thr_internal": float(results["best_f1_thr"]),
        "best_f1_internal": float(results["best_f1_stats"]["f1"]),

        "best_f2_thr_internal": float(results["best_f2_thr"]),
        "best_f2_internal": float(results["best_f2_stats"]["f2"]),

        "best_prec_thr_recall85_internal": float(results["best_prec_thr_recall85"]),
        "best_prec_recall85_internal": float(results["best_prec_stats_recall85"]["precision"]),
        "best_prec_recall85_recall_internal": float(results["best_prec_stats_recall85"]["recall"]),
        "best_prec_recall85_f2_internal": float(results["best_prec_stats_recall85"]["f2"]),
    }
    return summary


# -----------------------------
# Generate Summaries at Three Operating Points
# -----------------------------

# Operating point A: Best F1 threshold from validation
# Balanced precision and recall
val_summary_f1 = summarize_eval_results("val", val_results, val_thr_f1)
test_summary_f1 = summarize_eval_results("test", test_results, val_thr_f1)
ca_summary_f1 = summarize_eval_results("central_asia", ca_results, val_thr_f1) if ca_results is not None else None

# Operating point B: Best F2 threshold from validation
# Emphasizes recall over precision (2x weight on recall)
val_summary_f2 = summarize_eval_results("val", val_results, val_thr_f2)
test_summary_f2 = summarize_eval_results("test", test_results, val_thr_f2)
ca_summary_f2 = summarize_eval_results("central_asia", ca_results, val_thr_f2) if ca_results is not None else None

# Operating point C: Best precision at recall ≥ 0.85
# High recall guarantee with best possible precision
val_summary_prec85 = summarize_eval_results("val", val_results, val_thr_prec85)
test_summary_prec85 = summarize_eval_results("test", test_results, val_thr_prec85)
ca_summary_prec85 = summarize_eval_results("central_asia", ca_results, val_thr_prec85) if ca_results is not None else None


# -----------------------------
# Print Summaries
# -----------------------------

def print_summary(summary, title):
    """Print formatted summary table."""
    if summary is None:
        return
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    for k in [
        "loss", "auc", "ap", "report_threshold",
        "precision", "recall", "specificity", "accuracy",
        "f1", "f2", "tp", "tn", "fp", "fn"
    ]:
        print(f"{k:18s}: {summary[k]}")

# Print all summaries for three operating points
print_summary(val_summary_f1, "VAL SUMMARY @ VAL BEST-F1 THRESHOLD")
print_summary(test_summary_f1, "TEST SUMMARY @ VAL BEST-F1 THRESHOLD")
if ca_summary_f1 is not None:
    print_summary(ca_summary_f1, "CENTRAL ASIA SUMMARY @ VAL BEST-F1 THRESHOLD")

print_summary(val_summary_f2, "VAL SUMMARY @ VAL BEST-F2 THRESHOLD")
print_summary(test_summary_f2, "TEST SUMMARY @ VAL BEST-F2 THRESHOLD")
if ca_summary_f2 is not None:
    print_summary(ca_summary_f2, "CENTRAL ASIA SUMMARY @ VAL BEST-F2 THRESHOLD")

print_summary(val_summary_prec85, "VAL SUMMARY @ VAL BEST-PRECISION(Recall>=0.85) THRESHOLD")
print_summary(test_summary_prec85, "TEST SUMMARY @ VAL BEST-PRECISION(Recall>=0.85) THRESHOLD")
if ca_summary_prec85 is not None:
    print_summary(ca_summary_prec85, "CENTRAL ASIA SUMMARY @ VAL BEST-PRECISION(Recall>=0.85) THRESHOLD")


# -----------------------------
# Save Prediction CSVs
# -----------------------------

val_pred_csv = os.path.join(LOCAL_REPORTS, f"{STAGE1_RUN_NAME}_val_predictions.csv")
test_pred_csv = os.path.join(LOCAL_REPORTS, f"{STAGE1_RUN_NAME}_test_predictions.csv")
ca_pred_csv = os.path.join(LOCAL_REPORTS, f"{STAGE1_RUN_NAME}_central_asia_predictions.csv")


def attach_threshold_columns(pred_df, thr_f1, thr_f2, thr_prec85):
    """
    Add binary prediction columns for each operating threshold.

    This allows users to easily filter predictions at different
    precision/recall tradeoffs without recomputing.
    """
    out = pred_df.copy()
    out["pred_at_val_best_f1"] = (out["y_prob"].values >= thr_f1).astype(int)
    out["pred_at_val_best_f2"] = (out["y_prob"].values >= thr_f2).astype(int)
    out["pred_at_val_best_prec_recall85"] = (out["y_prob"].values >= thr_prec85).astype(int)
    return out


# Export predictions with threshold columns
val_pred_df_export = attach_threshold_columns(val_results["pred_df"], val_thr_f1, val_thr_f2, val_thr_prec85)
test_pred_df_export = attach_threshold_columns(test_results["pred_df"], val_thr_f1, val_thr_f2, val_thr_prec85)
val_pred_df_export.to_csv(val_pred_csv, index=False)
test_pred_df_export.to_csv(test_pred_csv, index=False)

if ca_results is not None:
    ca_pred_df_export = attach_threshold_columns(ca_results["pred_df"], val_thr_f1, val_thr_f2, val_thr_prec85)
    ca_pred_df_export.to_csv(ca_pred_csv, index=False)

print("\nSaved prediction CSVs:")
print(val_pred_csv)
print(test_pred_csv)
if ca_results is not None:
    print(ca_pred_csv)


# -----------------------------
# Save Metrics JSON
# -----------------------------

# Comprehensive metrics package for programmatic access
metrics_payload = {
    "run_name": STAGE1_RUN_NAME,
    "best_checkpoint": BEST_CKPT_PATH,
    "last_checkpoint": LAST_CKPT_PATH,
    "history_json": HISTORY_JSON_PATH,
    "best_epoch": best_payload.get("epoch"),
    "best_metric_name": saved_best_metric_name,
    "best_metric_value": best_payload.get("best_metric"),

    # Validation-selected thresholds (applied to all datasets)
    "val_thresholds": {
        "best_f1_threshold": float(val_thr_f1),
        "best_f2_threshold": float(val_thr_f2),
        "best_precision_recall85_threshold": float(val_thr_prec85),
    },

    # Results at each operating point
    "summaries": {
        "val_at_val_best_f1": val_summary_f1,
        "test_at_val_best_f1": test_summary_f1,
        "val_at_val_best_f2": val_summary_f2,
        "test_at_val_best_f2": test_summary_f2,
        "val_at_val_best_prec85": val_summary_prec85,
        "test_at_val_best_prec85": test_summary_prec85,
    }
}

# Add Central Asia results if available
if ca_summary_f1 is not None:
    metrics_payload["summaries"]["central_asia_at_val_best_f1"] = ca_summary_f1
if ca_summary_f2 is not None:
    metrics_payload["summaries"]["central_asia_at_val_best_f2"] = ca_summary_f2
if ca_summary_prec85 is not None:
    metrics_payload["summaries"]["central_asia_at_val_best_prec85"] = ca_summary_prec85

metrics_json_path = os.path.join(LOCAL_REPORTS, f"{STAGE1_RUN_NAME}_metrics.json")
with open(metrics_json_path, "w") as f:
    json.dump(metrics_payload, f, indent=2)

print("\nSaved metrics JSON:")
print(metrics_json_path)


# -----------------------------
# Save Markdown Report
# -----------------------------

report_md_path = os.path.join(LOCAL_REPORTS, f"{STAGE1_RUN_NAME}_report.md")


def make_md_table(summary):
    """Generate markdown table from summary dictionary."""
    if summary is None:
        return "_Not available_\n"
    keys = [
        "loss", "auc", "ap", "report_threshold",
        "precision", "recall", "specificity", "accuracy",
        "f1", "f2", "tp", "tn", "fp", "fn"
    ]
    lines = ["| metric | value |", "|---|---:|"]
    for k in keys:
        v = summary[k]
        if isinstance(v, float):
            lines.append(f"| {k} | {v:.6f} |")
        else:
            lines.append(f"| {k} | {v} |")
    return "\n".join(lines) + "\n"


# Build markdown report
md = []
md.append(f"# Stage 1 Classification Report — {STAGE1_RUN_NAME}\n")

# Artifacts section
md.append("## Artifacts\n")
md.append(f"- Best checkpoint: `{BEST_CKPT_PATH}`")
md.append(f"- Last checkpoint: `{LAST_CKPT_PATH}`")
md.append(f"- History JSON: `{HISTORY_JSON_PATH}`")
md.append(f"- Metrics JSON: `{metrics_json_path}`")
md.append(f"- Val predictions: `{val_pred_csv}`")
md.append(f"- Test predictions: `{test_pred_csv}`")
if ca_results is not None:
    md.append(f"- Central Asia predictions: `{ca_pred_csv}`")

# Thresholds section
md.append("\n## Validation-selected thresholds\n")
md.append(f"- Best F1 threshold on val: `{val_thr_f1:.6f}`")
md.append(f"- Best F2 threshold on val: `{val_thr_f2:.6f}`")
md.append(f"- Best precision with recall>=0.85 on val: `{val_thr_prec85:.6f}`")

# Results at best F1 threshold
md.append("\n## Results @ Val Best-F1 Threshold\n")
md.append("\n### Val\n")
md.append(make_md_table(val_summary_f1))
md.append("\n### Test\n")
md.append(make_md_table(test_summary_f1))
if ca_summary_f1 is not None:
    md.append("\n### Central Asia\n")
    md.append(make_md_table(ca_summary_f1))

# Results at best F2 threshold
md.append("\n## Results @ Val Best-F2 Threshold\n")
md.append("\n### Val\n")
md.append(make_md_table(val_summary_f2))
md.append("\n### Test\n")
md.append(make_md_table(test_summary_f2))
if ca_summary_f2 is not None:
    md.append("\n### Central Asia\n")
    md.append(make_md_table(ca_summary_f2))

# Results at best precision (recall >= 0.85) threshold
md.append("\n## Results @ Val Best-Precision(Recall>=0.85) Threshold\n")
md.append("\n### Val\n")
md.append(make_md_table(val_summary_prec85))
md.append("\n### Test\n")
md.append(make_md_table(test_summary_prec85))
if ca_summary_prec85 is not None:
    md.append("\n### Central Asia\n")
    md.append(make_md_table(ca_summary_prec85))

# Write markdown report
with open(report_md_path, "w") as f:
    f.write("\n".join(md))

print("\nSaved markdown report:")
print(report_md_path)

print("\n" + "=" * 70)
print("FINAL EVALUATION COMPLETE")
print("=" * 70)
print("All metrics, predictions, and reports have been exported.")
print("=" * 70)

## Optional: FEW-SHOT TRANSFER LEARNING EXPERIMENT

In [ ]:
# ============================================================
# FEW-SHOT TRANSFER LEARNING EXPERIMENT
# ============================================================
# This section evaluates how well the Amazon-trained model transfers to
# Central Asia with limited labeled data (few-shot learning scenario).
#
# EXPERIMENT DESIGN:
# - Load best Amazon checkpoint as starting point
# - Fine-tune on varying amounts of Central Asia data: 1%, 10%, 25%, 50%
# - Evaluate all fractions on the SAME shared holdout set (fair comparison)
# - Each fraction uses its OWN normalization statistics (realistic few-shot)
#
# KEY DESIGN DECISIONS:
# 1. Shared holdout: All fractions evaluated on identical samples
# 2. Per-fraction normalization: Each subset computes own channel stats
# 3. Partial model unfreezing: Only train classifier + last N feature layers
# 4. Differential learning rates: Lower for backbone, higher for head
# 5. Robust sampling: Ensures class balance even with tiny datasets
# 6. Numerical stability: Heavy sanitization for small-batch training
#
# WHY THIS MATTERS:
# - Tests cross-geography generalization (Amazon → Central Asia)
# - Measures label efficiency (how much CA data is needed)
# - Realistic scenario: new regions have limited labeled examples
#
# DESIGN NOTE: This section can be re-run independently after the main
# training pipeline completes, but requires the following from earlier cells:
# - Trained model and BEST_CKPT_PATH
# - Central Asia data (central_asia_eval_labeled_df)
# - Core data loading functions (read_sample_raw_from_row, raw_to_physical)
# The precondition check below verifies these dependencies exist.
# All metric/normalization helpers are reimplemented here for self-containment.
# ============================================================

import os
import json
import copy
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# Preconditions Check
# ------------------------------------------------------------
# Verify all required variables exist from previous cells

required_names = [
    "SEED",
    "device",
    "model",
    "BATCH_SIZE",
    "NUM_WORKERS",
    "LOCAL_CKPT",
    "LOCAL_REPORTS",
    "BEST_CKPT_PATH",
    "STAGE1_RUN_NAME",
    "CHANNEL_ORDER",
    "central_asia_eval_labeled_df",
    "read_sample_raw_from_row",
    "raw_to_physical",
]
missing = [x for x in required_names if x not in globals()]
if missing:
    raise NameError("Missing required names:\n" + "\n".join(missing))

os.makedirs(LOCAL_CKPT, exist_ok=True)
os.makedirs(LOCAL_REPORTS, exist_ok=True)

# ------------------------------------------------------------
# Few-Shot Experiment Configuration
# ------------------------------------------------------------

FS_RUN_NAME = f"{STAGE1_RUN_NAME}_ca_fewshot_shared_holdout_ca_norm"

# Data split ratios (for outer CA train/val/test)
FS_VAL_FRAC = 0.15
FS_TEST_FRAC = 0.15

# Shared holdout: carved from CA-train for fair cross-fraction comparison
# This ensures all fractions are evaluated on exactly the same samples
FS_SHARED_HOLDOUT_FRAC_WITHIN_TRAIN = 0.20

# Few-shot fractions to test
# Each represents percentage of available CA training groups
FS_FRACTIONS = [0.01, 0.10, 0.25, 0.50]

# Training configuration
FS_NUM_EPOCHS = 6  # Fixed epochs for all fractions (for fair comparison)

# Differential learning rates (critical for transfer learning)
FS_LR_HEAD = 5e-5       # Higher LR for classifier head (learning new task)
FS_LR_BACKBONE = 5e-6   # Lower LR for backbone (preserve Amazon features)
FS_WEIGHT_DECAY = 1e-4

# Partial unfreezing strategy
# Only train the classifier and last N feature blocks to prevent overfitting
FS_UNFREEZE_LAST_N_FEATURE_MODULES = 2

# Numerical stability settings (critical for small-batch training)
FS_INPUT_CLIP_VALUE = 20.0      # Clip input z-scores to prevent explosions
FS_GRAD_CLIP_NORM = 1.0         # Gradient clipping for stability
FS_NORM_PIXELS_PER_IMAGE = 4096 # Pixels to sample for normalization stats

# Loss configuration
FS_POS_WEIGHT = 2.0              # Weight positive class more (recall emphasis)
FS_TARGET_MIN_RECALL = 0.85      # Target recall for threshold selection

# Robust sampling constraints
# Ensures even 1% samples have both classes and aren't too imbalanced
FS_MIN_GROUPS = 2           # Minimum number of groups to sample
FS_MIN_POS = 1              # Minimum positive samples required
FS_MIN_NEG = 1              # Minimum negative samples required
FS_MAX_SAMPLING_TRIES = 300 # Max attempts to find balanced sample

# Output paths
FS_SHARED_HOLDOUT_CSV = os.path.join(LOCAL_REPORTS, f"{FS_RUN_NAME}_shared_holdout_eval.csv")
FS_SUMMARY_CSV = os.path.join(LOCAL_REPORTS, f"{FS_RUN_NAME}_summary.csv")
FS_HISTORY_CSV = os.path.join(LOCAL_REPORTS, f"{FS_RUN_NAME}_history.csv")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Helper Functions (reimplemented for independence)
# ------------------------------------------------------------

def safe_roc_auc(y_true, y_prob):
    """Compute ROC-AUC, return NaN if only one class present."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))


def safe_ap(y_true, y_prob):
    """Compute Average Precision, return NaN if only one class present."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))


def fbeta_from_pr(precision, recall, beta=1.0):
    """Compute F-beta score from precision and recall."""
    denom = (beta**2 * precision + recall)
    if denom <= 0:
        return 0.0
    return (1 + beta**2) * precision * recall / denom


def compute_threshold_metrics(y_true, y_prob, threshold=0.5):
    """
    Compute classification metrics at a specific threshold.
    Includes sanitization for numerical stability.
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(np.float32)

    # Sanitize probabilities
    y_prob = np.nan_to_num(y_prob, nan=0.5, posinf=1.0, neginf=0.0)
    y_prob = np.clip(y_prob, 0.0, 1.0)

    y_pred = (y_prob >= threshold).astype(int)

    # Confusion matrix components
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    # Derived metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    accuracy = (tp + tn) / max(1, len(y_true))
    f1 = fbeta_from_pr(precision, recall, beta=1.0)
    f2 = fbeta_from_pr(precision, recall, beta=2.0)

    return {
        "threshold": float(threshold),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "accuracy": float(accuracy),
        "f1": float(f1),
        "f2": float(f2),
    }


def find_best_fbeta_threshold(y_true, y_prob, beta=1.0):
    """Find threshold that maximizes F-beta score."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(np.float32)
    y_prob = np.nan_to_num(y_prob, nan=0.5, posinf=1.0, neginf=0.0)
    y_prob = np.clip(y_prob, 0.0, 1.0)

    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    if len(thresholds) == 0:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
        return 0.5, stats

    best_thr = 0.5
    best_score = -1.0
    best_stats = None

    for thr in thresholds:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=float(thr))
        score = stats["f2"] if beta == 2.0 else stats["f1"] if beta == 1.0 else fbeta_from_pr(
            stats["precision"], stats["recall"], beta=beta
        )
        if score > best_score:
            best_score = score
            best_thr = float(thr)
            best_stats = stats.copy()

    return best_thr, best_stats


def find_best_f1_threshold(y_true, y_prob):
    """Find threshold that maximizes F1 score."""
    return find_best_fbeta_threshold(y_true, y_prob, beta=1.0)


def find_best_f2_threshold(y_true, y_prob):
    """Find threshold that maximizes F2 score."""
    return find_best_fbeta_threshold(y_true, y_prob, beta=2.0)


def find_best_precision_under_recall(y_true, y_prob, min_recall=0.85):
    """
    Find threshold that maximizes precision while maintaining minimum recall.
    Critical for archaeological detection where missing sites is costly.
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(np.float32)
    y_prob = np.nan_to_num(y_prob, nan=0.5, posinf=1.0, neginf=0.0)
    y_prob = np.clip(y_prob, 0.0, 1.0)

    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    if len(thresholds) == 0:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
        return 0.5, stats

    best_thr = None
    best_stats = None
    best_precision = -1.0

    for thr in thresholds:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=float(thr))
        if stats["recall"] >= min_recall and stats["precision"] > best_precision:
            best_precision = stats["precision"]
            best_thr = float(thr)
            best_stats = stats.copy()

    if best_thr is None:
        stats = compute_threshold_metrics(y_true, y_prob, threshold=0.5)
        return 0.5, stats

    return best_thr, best_stats


def _sanitize_tensor(x, clip_value=20.0):
    """
    Sanitize tensor for numerical stability.
    Critical for small-batch training where outliers can destabilize training.
    """
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = torch.clamp(x, -clip_value, clip_value)
    return x


def _sanitize_numpy_prob(x):
    """Sanitize probability array to valid [0, 1] range."""
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.5, posinf=1.0, neginf=0.0)
    x = np.clip(x, 0.0, 1.0)
    return x


def sample_pixels_1d(arr2d, k=4096, rng=None):
    """Sample k random pixels from 2D array for normalization statistics."""
    flat = np.asarray(arr2d).reshape(-1)
    n = flat.shape[0]
    if n <= k:
        return flat.astype(np.float32)
    idx = rng.choice(n, size=k, replace=False)
    return flat[idx].astype(np.float32)


def build_norm_stats_from_train_df(df_train, pixels_per_image=4096):
    """
    Compute normalization statistics from few-shot training data.

    CRITICAL: Each fraction computes its OWN statistics from its limited data.
    This is realistic - in true few-shot scenarios, you only have the few
    labeled examples to compute normalization from.
    """
    rng = np.random.default_rng(SEED)
    ch_vals = {ch: [] for ch in CHANNEL_ORDER}

    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Fit few-shot norm stats"):
        x_raw = read_sample_raw_from_row(row)
        x_phys = raw_to_physical(x_raw).astype(np.float32)
        x_phys = np.nan_to_num(x_phys, nan=0.0, posinf=0.0, neginf=0.0)

        for ci, ch in enumerate(CHANNEL_ORDER):
            vals = sample_pixels_1d(x_phys[ci], k=pixels_per_image, rng=rng)
            ch_vals[ch].append(vals)

    norm_stats = {}
    for ch in CHANNEL_ORDER:
        vals = np.concatenate(ch_vals[ch]).astype(np.float32)
        norm_stats[ch] = {
            "mean": float(np.mean(vals)),
            "std": max(float(np.std(vals)), 1e-6),  # Prevent division by zero
        }
    return norm_stats


def normalize_from_stats_dict(x_raw, stats_dict):
    """Normalize sample using provided statistics dictionary."""
    x_phys = raw_to_physical(x_raw).astype(np.float32)
    x_phys = np.nan_to_num(x_phys, nan=0.0, posinf=0.0, neginf=0.0)

    out = np.empty_like(x_phys, dtype=np.float32)
    for ci, ch in enumerate(CHANNEL_ORDER):
        mu = float(stats_dict[ch]["mean"])
        sd = max(float(stats_dict[ch]["std"]), 1e-6)
        out[ci] = ((x_phys[ci] - mu) / sd).astype(np.float32)
    return out


def has_both_classes(df, label_col="class_label"):
    """Check if DataFrame contains both positive and negative samples."""
    vals = pd.Series(df[label_col]).dropna().astype(int).unique().tolist()
    return (0 in vals) and (1 in vals)


def assert_both_classes(df, name, label_col="class_label"):
    """
    Assert DataFrame has both classes, raise informative error if not.
    Critical for few-shot learning where tiny samples might miss a class.
    """
    vals = sorted(pd.Series(df[label_col]).dropna().astype(int).unique().tolist())
    if not ((0 in vals) and (1 in vals)):
        raise ValueError(
            f"{name} does not contain both classes. "
            f"Found labels={vals}, rows={len(df)}"
        )


class CAFewShotDataset(Dataset):
    """
    Dataset for Central Asia few-shot fine-tuning.
    Uses provided normalization statistics (computed from few-shot train set).
    """

    def __init__(self, df, stats_dict, return_metadata=True):
        self.df = df.reset_index(drop=True)
        self.stats_dict = stats_dict
        self.return_metadata = return_metadata

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x_raw = read_sample_raw_from_row(row)
        x = normalize_from_stats_dict(x_raw, self.stats_dict)
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        y = np.float32(row["class_label"])

        meta = {}
        if self.return_metadata:
            for col in self.df.columns:
                val = row[col]
                if isinstance(val, (str, int, float, np.integer, np.floating, bool)) or pd.isna(val):
                    meta[col] = val

        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32), meta


def simple_collate_with_meta(batch):
    """Collate function that handles metadata."""
    xs, ys, metas = zip(*batch)
    return torch.stack(xs, dim=0), torch.stack(ys, dim=0), list(metas)


def build_weighted_sampler(df):
    """
    Create weighted sampler for class balance.
    Critical for few-shot learning where class imbalance is common.
    """
    labels = df["class_label"].values.astype(int)
    class_counts = np.bincount(labels, minlength=2)

    if class_counts[0] == 0 or class_counts[1] == 0:
        raise ValueError(
            f"Weighted sampler requires both classes. "
            f"class_counts={class_counts.tolist()}"
        )

    class_weights = np.zeros_like(class_counts, dtype=np.float64)
    for c in [0, 1]:
        class_weights[c] = 1.0 / class_counts[c]

    sample_weights = np.array([class_weights[y] for y in labels], dtype=np.float64)
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True
    )


def make_model_fresh_from_checkpoint():
    """
    Load model from Amazon checkpoint and configure for few-shot fine-tuning.

    STRATEGY:
    - Freeze most of the backbone (preserve Amazon features)
    - Unfreeze classifier head (learn CA-specific decision boundary)
    - Unfreeze last N feature blocks (allow subtle adaptation)
    - Set BatchNorm to eval for frozen layers, train for unfrozen

    This prevents overfitting on tiny CA datasets while allowing adaptation.
    """
    if not os.path.exists(BEST_CKPT_PATH):
        raise FileNotFoundError(f"BEST_CKPT_PATH not found: {BEST_CKPT_PATH}")

    payload = torch.load(BEST_CKPT_PATH, map_location=device)
    if isinstance(payload, dict) and "model_state_dict" in payload:
        model.load_state_dict(payload["model_state_dict"])
    else:
        model.load_state_dict(payload)

    model.to(device)

    # Freeze all parameters initially
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze classifier head (always trainable)
    for p in model.classifier.parameters():
        p.requires_grad = True

    # Unfreeze last N feature blocks for subtle adaptation
    n_feat_modules = len(model.features)
    start_idx = max(0, n_feat_modules - FS_UNFREEZE_LAST_N_FEATURE_MODULES)
    for i in range(start_idx, n_feat_modules):
        for p in model.features[i].parameters():
            p.requires_grad = True

    # Set BatchNorm mode: eval for frozen, train for unfrozen
    def set_bn_mode(module):
        if isinstance(module, nn.BatchNorm2d):
            any_trainable = any(p.requires_grad for p in module.parameters())
            module.train(any_trainable)

    model.apply(set_bn_mode)
    return model


def train_one_epoch_ft(model, loader, optimizer, criterion, device):
    """
    Train one epoch with numerical stability checks.
    Skips batches with NaN/Inf to prevent training collapse.
    """
    model.train()
    running_loss = 0.0
    all_y = []
    all_prob = []
    n_seen = 0
    n_skipped_bad_batch = 0

    for xb, yb, _ in tqdm(loader, total=len(loader), leave=False):
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float()

        # Sanitize inputs for stability
        xb = _sanitize_tensor(xb, clip_value=FS_INPUT_CLIP_VALUE)
        yb = torch.nan_to_num(yb, nan=0.0, posinf=1.0, neginf=0.0)

        optimizer.zero_grad(set_to_none=True)

        logits = model(xb).view(-1)

        # Skip batch if logits are invalid
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            n_skipped_bad_batch += 1
            continue

        loss = criterion(logits, yb)

        # Skip batch if loss is invalid
        if torch.isnan(loss) or torch.isinf(loss):
            n_skipped_bad_batch += 1
            continue

        loss.backward()

        # Gradient clipping prevents explosions with small batches
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=FS_GRAD_CLIP_NORM)

        optimizer.step()

        # Collect predictions
        probs = torch.sigmoid(logits.detach()).float().cpu().numpy()
        probs = _sanitize_numpy_prob(probs)
        y_np = yb.detach().cpu().numpy().astype(np.float32)
        y_np = np.nan_to_num(y_np, nan=0.0, posinf=1.0, neginf=0.0)

        all_prob.append(probs)
        all_y.append(y_np)

        bs = xb.size(0)
        running_loss += float(loss.item()) * bs
        n_seen += bs

    if len(all_y) == 0:
        raise RuntimeError("All training batches became invalid.")

    # Compute epoch metrics
    y_true = np.concatenate(all_y)
    y_prob = np.concatenate(all_prob)
    y_true = np.nan_to_num(y_true, nan=0.0, posinf=1.0, neginf=0.0)
    y_prob = _sanitize_numpy_prob(y_prob)

    avg_loss = running_loss / max(1, n_seen)
    auc = safe_roc_auc(y_true, y_prob)
    ap = safe_ap(y_true, y_prob)
    best_f1_thr, best_f1_stats = find_best_f1_threshold(y_true, y_prob)
    best_f2_thr, best_f2_stats = find_best_f2_threshold(y_true, y_prob)
    best_prec_thr85, best_prec_stats85 = find_best_precision_under_recall(
        y_true, y_prob, min_recall=FS_TARGET_MIN_RECALL
    )

    return {
        "loss": float(avg_loss),
        "auc": float(auc) if np.isfinite(auc) else np.nan,
        "ap": float(ap) if np.isfinite(ap) else np.nan,
        "best_f1_thr": float(best_f1_thr),
        "best_f1_stats": best_f1_stats,
        "best_f2_thr": float(best_f2_thr),
        "best_f2_stats": best_f2_stats,
        "best_prec_thr_recall85": float(best_prec_thr85),
        "best_prec_stats_recall85": best_prec_stats85,
    }


@torch.no_grad()
def evaluate_one_epoch_ft(model, loader, criterion, device):
    """Evaluate one epoch with numerical stability."""
    model.eval()
    running_loss = 0.0
    all_y = []
    all_prob = []
    n_seen = 0

    for xb, yb, _ in tqdm(loader, total=len(loader), leave=False):
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float()

        xb = _sanitize_tensor(xb, clip_value=FS_INPUT_CLIP_VALUE)
        yb = torch.nan_to_num(yb, nan=0.0, posinf=1.0, neginf=0.0)

        logits = model(xb).view(-1)
        logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits).float().cpu().numpy()
        probs = _sanitize_numpy_prob(probs)
        y_np = yb.cpu().numpy().astype(np.float32)
        y_np = np.nan_to_num(y_np, nan=0.0, posinf=1.0, neginf=0.0)

        all_prob.append(probs)
        all_y.append(y_np)

        bs = xb.size(0)
        running_loss += float(loss.item()) * bs
        n_seen += bs

    if len(all_y) == 0:
        raise RuntimeError("All eval batches became invalid.")

    y_true = np.concatenate(all_y)
    y_prob = np.concatenate(all_prob)
    y_true = np.nan_to_num(y_true, nan=0.0, posinf=1.0, neginf=0.0)
    y_prob = _sanitize_numpy_prob(y_prob)

    avg_loss = running_loss / max(1, n_seen)
    auc = safe_roc_auc(y_true, y_prob)
    ap = safe_ap(y_true, y_prob)
    best_f1_thr, best_f1_stats = find_best_f1_threshold(y_true, y_prob)
    best_f2_thr, best_f2_stats = find_best_f2_threshold(y_true, y_prob)
    best_prec_thr85, best_prec_stats85 = find_best_precision_under_recall(
        y_true, y_prob, min_recall=FS_TARGET_MIN_RECALL
    )

    return {
        "loss": float(avg_loss),
        "auc": float(auc) if np.isfinite(auc) else np.nan,
        "ap": float(ap) if np.isfinite(ap) else np.nan,
        "y_true": y_true,
        "y_prob": y_prob,
        "best_f1_thr": float(best_f1_thr),
        "best_f1_stats": best_f1_stats,
        "best_f2_thr": float(best_f2_thr),
        "best_f2_stats": best_f2_stats,
        "best_prec_thr_recall85": float(best_prec_thr85),
        "best_prec_stats_recall85": best_prec_stats85,
    }


# ------------------------------------------------------------
# Create Fixed CA Train/Val/Test Split (Group-Aware)
# ------------------------------------------------------------

ca_df = central_asia_eval_labeled_df.copy()
ca_df["class_label"] = pd.to_numeric(ca_df["class_label"], errors="coerce")
ca_df = ca_df[ca_df["class_label"].isin([0, 1])].copy()
ca_df["class_label"] = ca_df["class_label"].astype(int)

# Determine grouping column (prevents data leakage across splits)
if "group_id" in ca_df.columns and ca_df["group_id"].notna().any():
    group_col = "group_id"
elif "site_id" in ca_df.columns and ca_df["site_id"].notna().any():
    group_col = "site_id"
else:
    group_col = "sample_id"

ca_df["_fs_group"] = ca_df[group_col].astype(str)

# Split 1: Train-full vs (Val + Test)
gss1 = GroupShuffleSplit(n_splits=1, test_size=FS_VAL_FRAC + FS_TEST_FRAC, random_state=SEED)
idx_train, idx_holdout = next(gss1.split(ca_df, groups=ca_df["_fs_group"]))

ca_train_full_df = ca_df.iloc[idx_train].copy().reset_index(drop=True)
ca_holdout_df = ca_df.iloc[idx_holdout].copy().reset_index(drop=True)

# Split 2: Val vs Test
holdout_test_frac = FS_TEST_FRAC / (FS_VAL_FRAC + FS_TEST_FRAC)
gss2 = GroupShuffleSplit(n_splits=1, test_size=holdout_test_frac, random_state=SEED)
idx_val, idx_test = next(gss2.split(ca_holdout_df, groups=ca_holdout_df["_fs_group"]))

ca_val_df = ca_holdout_df.iloc[idx_val].copy().reset_index(drop=True)
ca_test_df = ca_holdout_df.iloc[idx_test].copy().reset_index(drop=True)

# Verify all splits have both classes
assert_both_classes(ca_train_full_df, "ca_train_full_df")
assert_both_classes(ca_val_df, "ca_val_df")
assert_both_classes(ca_test_df, "ca_test_df")

print("Fixed outer split ready.")
print("CA train full / val / test rows:", len(ca_train_full_df), len(ca_val_df), len(ca_test_df))
print("Group split column:", group_col)


# ------------------------------------------------------------
# Create Shared Holdout Set (for fair cross-fraction comparison)
# ------------------------------------------------------------
# CRITICAL DESIGN: All fractions are evaluated on the SAME holdout samples.
# This ensures fair comparison - differences in performance are due to
# training set size, not evaluation set variation.

gss_shared = GroupShuffleSplit(
    n_splits=1,
    test_size=FS_SHARED_HOLDOUT_FRAC_WITHIN_TRAIN,
    random_state=SEED
)
idx_train_pool, idx_shared_holdout = next(
    gss_shared.split(ca_train_full_df, groups=ca_train_full_df["_fs_group"])
)

ca_train_pool_df = ca_train_full_df.iloc[idx_train_pool].copy().reset_index(drop=True)
ca_shared_holdout_df = ca_train_full_df.iloc[idx_shared_holdout].copy().reset_index(drop=True)

assert_both_classes(ca_train_pool_df, "ca_train_pool_df")
assert_both_classes(ca_shared_holdout_df, "ca_shared_holdout_df")

print("\nShared internal holdout ready.")
print("Train pool rows:", len(ca_train_pool_df))
print("Shared holdout rows:", len(ca_shared_holdout_df))
print("Train pool groups:", ca_train_pool_df["_fs_group"].nunique())
print("Shared holdout groups:", ca_shared_holdout_df["_fs_group"].nunique())

ca_shared_holdout_df.to_csv(FS_SHARED_HOLDOUT_CSV, index=False)
print("Saved shared holdout CSV:", FS_SHARED_HOLDOUT_CSV)


# ------------------------------------------------------------
# Few-Shot Sampling Function
# ------------------------------------------------------------

def sample_train_groups_fraction(
    df_train_pool,
    frac,
    seed=42,
    max_tries=300,
    min_groups=2,
    min_pos=1,
    min_neg=1,
    max_imbalance_ratio=3.0
):
    """
    Sample a fraction of groups with class balance constraints.

    CRITICAL: With tiny fractions (e.g., 1%), random sampling might produce
    all-positive or all-negative subsets, or extreme imbalance. This function
    retries until it finds a valid balanced sample.

    Args:
        df_train_pool: Available training data
        frac: Fraction of groups to sample (e.g., 0.01 = 1%)
        seed: Random seed
        max_tries: Maximum sampling attempts before failure
        min_groups: Minimum number of groups required
        min_pos: Minimum positive samples required
        min_neg: Minimum negative samples required
        max_imbalance_ratio: Max allowed pos:neg or neg:pos ratio

    Returns:
        out: Sampled DataFrame
        chosen: List of chosen group IDs
    """
    groups = pd.Series(df_train_pool["_fs_group"].unique())
    n_groups = len(groups)
    take_n = max(min_groups, int(round(frac * n_groups)))
    take_n = min(take_n, n_groups)

    rng = np.random.RandomState(seed)

    for attempt in range(max_tries):
        chosen = groups.sample(
            n=take_n,
            random_state=int(rng.randint(0, 1_000_000))
        ).tolist()

        out = df_train_pool[df_train_pool["_fs_group"].isin(chosen)].copy().reset_index(drop=True)

        n_pos = int((out["class_label"] == 1).sum())
        n_neg = int((out["class_label"] == 0).sum())

        # Check 1: Both classes must exist
        if n_pos < min_pos or n_neg < min_neg:
            continue

        # Check 2: Classes should not be too imbalanced
        ratio = max(n_pos, n_neg) / max(1, min(n_pos, n_neg))
        if ratio <= max_imbalance_ratio:
            print(f"  ✓ Sampled: {n_pos} pos / {n_neg} neg (ratio {ratio:.2f}:1)")
            return out, chosen

    # If failed after max_tries, provide detailed error
    raise RuntimeError(
        f"Failed to sample few-shot subset after {max_tries} attempts.\n"
        f"Requirements: frac={frac}, min_pos={min_pos}, min_neg={min_neg}, "
        f"max_imbalance_ratio={max_imbalance_ratio}\n"
        f"Try increasing max_tries or relaxing max_imbalance_ratio."
    )


# ------------------------------------------------------------
# Main Few-Shot Learning Curve Loop
# ------------------------------------------------------------

summary_rows = []
history_rows = []

for frac in FS_FRACTIONS:
    print("\n" + "=" * 80)
    print(f"FEW-SHOT FRACTION = {frac:.0%}")
    print("=" * 80)

    # Sample few-shot training subset
    ca_train_df, chosen_groups = sample_train_groups_fraction(
        ca_train_pool_df,
        frac=frac,
        seed=SEED,
        max_tries=FS_MAX_SAMPLING_TRIES,
        min_groups=FS_MIN_GROUPS,
        min_pos=FS_MIN_POS,
        min_neg=FS_MIN_NEG,
    )

    assert_both_classes(ca_train_df, f"few-shot train subset @ {frac:.0%}")

    print(
        f"Sampled train subset: rows={len(ca_train_df)}, "
        f"groups={ca_train_df['_fs_group'].nunique()}, "
        f"pos={(ca_train_df['class_label'] == 1).sum()}, "
        f"neg={(ca_train_df['class_label'] == 0).sum()}"
    )

    # Compute normalization stats from THIS few-shot subset only
    # This simulates realistic few-shot scenario where you only have these samples
    ca_train_norm_stats = build_norm_stats_from_train_df(
        ca_train_df,
        pixels_per_image=FS_NORM_PIXELS_PER_IMAGE
    )

    # Create datasets using few-shot normalization
    train_ds = CAFewShotDataset(ca_train_df, stats_dict=ca_train_norm_stats, return_metadata=True)
    holdout_ds = CAFewShotDataset(ca_shared_holdout_df, stats_dict=ca_train_norm_stats, return_metadata=True)

    # Create DataLoaders
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=build_weighted_sampler(ca_train_df),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=simple_collate_with_meta,
        drop_last=False,
    )
    holdout_loader = DataLoader(
        holdout_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=simple_collate_with_meta,
        drop_last=False,
    )

    # Load fresh model from Amazon checkpoint
    model = make_model_fresh_from_checkpoint()

    # Sanity check: verify forward pass is stable before training
    model.eval()
    with torch.no_grad():
        xb0, yb0, _ = next(iter(train_loader))
        xb0 = _sanitize_tensor(xb0.to(device).float(), clip_value=FS_INPUT_CLIP_VALUE)
        logits0 = model(xb0).view(-1)
        if torch.isnan(logits0).any() or torch.isinf(logits0).any():
            raise RuntimeError(f"Forward unstable before training for fraction={frac}")

    # Setup optimizer with differential learning rates
    backbone_params = []
    head_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("classifier."):
            head_params.append(p)
        else:
            backbone_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": FS_LR_BACKBONE},  # Lower LR for backbone
            {"params": head_params, "lr": FS_LR_HEAD},          # Higher LR for head
        ],
        weight_decay=FS_WEIGHT_DECAY,
    )

    # Loss with positive class weighting
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([FS_POS_WEIGHT], device=device)
    )

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
    )

    # Track best model for this fraction
    best_holdout_metric = -np.inf
    best_epoch = -1
    best_payload = None

    # Training loop (fixed epochs for all fractions)
    for epoch in range(1, FS_NUM_EPOCHS + 1):
        train_res = train_one_epoch_ft(model, train_loader, optimizer, criterion, device)
        holdout_res = evaluate_one_epoch_ft(model, holdout_loader, criterion, device)

        current_holdout_metric = holdout_res["best_prec_stats_recall85"]["precision"]
        scheduler.step(current_holdout_metric if np.isfinite(current_holdout_metric) else -1e9)

        # Record training history
        history_rows.append({
            "fraction": frac,
            "epoch": epoch,
            "train_rows": len(ca_train_df),
            "train_groups": int(ca_train_df["_fs_group"].nunique()),
            "train_pos": int((ca_train_df["class_label"] == 1).sum()),
            "train_neg": int((ca_train_df["class_label"] == 0).sum()),
            "shared_holdout_rows": len(ca_shared_holdout_df),
            "shared_holdout_groups": int(ca_shared_holdout_df["_fs_group"].nunique()),

            "train_loss": train_res["loss"],
            "train_auc": train_res["auc"],
            "train_ap": train_res["ap"],
            "train_best_f1": train_res["best_f1_stats"]["f1"],
            "train_best_f2": train_res["best_f2_stats"]["f2"],
            "train_best_prec_recall85": train_res["best_prec_stats_recall85"]["precision"],
            "train_best_prec_recall85_recall": train_res["best_prec_stats_recall85"]["recall"],

            "holdout_loss": holdout_res["loss"],
            "holdout_auc": holdout_res["auc"],
            "holdout_ap": holdout_res["ap"],
            "holdout_best_f1": holdout_res["best_f1_stats"]["f1"],
            "holdout_best_f1_thr": holdout_res["best_f1_thr"],
            "holdout_best_f2": holdout_res["best_f2_stats"]["f2"],
            "holdout_best_f2_thr": holdout_res["best_f2_thr"],
            "holdout_best_prec_recall85": holdout_res["best_prec_stats_recall85"]["precision"],
            "holdout_best_prec_recall85_recall": holdout_res["best_prec_stats_recall85"]["recall"],
            "holdout_best_prec_thr_recall85": holdout_res["best_prec_thr_recall85"],
        })

        print(
            f"[{frac:.0%}] epoch={epoch} | "
            f"train_ap={train_res['ap']:.4f} | "
            f"holdout_ap={holdout_res['ap']:.4f} | "
            f"holdout_auc={holdout_res['auc']:.4f} | "
            f"holdout_prec@recall>=0.85={holdout_res['best_prec_stats_recall85']['precision']:.4f} "
            f"(recall={holdout_res['best_prec_stats_recall85']['recall']:.4f})"
        )

        # Save best checkpoint for this fraction
        improved = np.isfinite(current_holdout_metric) and (current_holdout_metric > best_holdout_metric)
        if improved:
            best_holdout_metric = current_holdout_metric
            best_epoch = epoch
            best_payload = {
                "epoch": epoch,
                "model_state_dict": copy.deepcopy(model.state_dict()),
                "holdout_best_f1_thr": float(holdout_res["best_f1_thr"]),
                "holdout_best_f2_thr": float(holdout_res["best_f2_thr"]),
                "holdout_best_prec_thr_recall85": float(holdout_res["best_prec_thr_recall85"]),
                "best_metric_name": "holdout_best_prec_recall85",
                "best_metric_value": float(best_holdout_metric),
            }

    if best_payload is None:
        raise RuntimeError(f"No valid checkpoint for fraction={frac}")

    # Evaluate best checkpoint on shared holdout
    model.load_state_dict(best_payload["model_state_dict"])
    model.eval()

    holdout_final = evaluate_one_epoch_ft(model, holdout_loader, criterion, device)

    # Extract thresholds from best checkpoint
    holdout_thr_f1 = best_payload["holdout_best_f1_thr"]
    holdout_thr_f2 = best_payload["holdout_best_f2_thr"]
    holdout_thr_prec85 = best_payload["holdout_best_prec_thr_recall85"]

    # Compute metrics at each threshold
    holdout_stats_f1 = compute_threshold_metrics(
        holdout_final["y_true"],
        holdout_final["y_prob"],
        threshold=holdout_thr_f1
    )
    holdout_stats_f2 = compute_threshold_metrics(
        holdout_final["y_true"],
        holdout_final["y_prob"],
        threshold=holdout_thr_f2
    )
    holdout_stats_prec85 = compute_threshold_metrics(
        holdout_final["y_true"],
        holdout_final["y_prob"],
        threshold=holdout_thr_prec85
    )

    # Record summary for this fraction
    summary_rows.append({
        "fraction": frac,
        "train_rows": len(ca_train_df),
        "train_groups": int(ca_train_df["_fs_group"].nunique()),
        "train_pos": int((ca_train_df["class_label"] == 1).sum()),
        "train_neg": int((ca_train_df["class_label"] == 0).sum()),
        "shared_holdout_rows": len(ca_shared_holdout_df),
        "shared_holdout_groups": int(ca_shared_holdout_df["_fs_group"].nunique()),
        "best_epoch": int(best_epoch),

        "holdout_auc": float(holdout_final["auc"]),
        "holdout_ap": float(holdout_final["ap"]),

        "holdout_best_f1_thr": float(holdout_thr_f1),
        "holdout_precision_at_best_f1": float(holdout_stats_f1["precision"]),
        "holdout_recall_at_best_f1": float(holdout_stats_f1["recall"]),
        "holdout_specificity_at_best_f1": float(holdout_stats_f1["specificity"]),
        "holdout_f1_at_best_f1": float(holdout_stats_f1["f1"]),
        "holdout_f2_at_best_f1": float(holdout_stats_f1["f2"]),

        "holdout_best_f2_thr": float(holdout_thr_f2),
        "holdout_precision_at_best_f2": float(holdout_stats_f2["precision"]),
        "holdout_recall_at_best_f2": float(holdout_stats_f2["recall"]),
        "holdout_specificity_at_best_f2": float(holdout_stats_f2["specificity"]),
        "holdout_f1_at_best_f2": float(holdout_stats_f2["f1"]),
        "holdout_f2_at_best_f2": float(holdout_stats_f2["f2"]),

        "holdout_best_prec_thr_recall85": float(holdout_thr_prec85),
        "holdout_precision_at_best_prec85": float(holdout_stats_prec85["precision"]),
        "holdout_recall_at_best_prec85": float(holdout_stats_prec85["recall"]),
        "holdout_specificity_at_best_prec85": float(holdout_stats_prec85["specificity"]),
        "holdout_f1_at_best_prec85": float(holdout_stats_prec85["f1"]),
        "holdout_f2_at_best_prec85": float(holdout_stats_prec85["f2"]),
        "holdout_accuracy_at_best_prec85": float(holdout_stats_prec85["accuracy"]),
        "holdout_tp_at_best_prec85": int(holdout_stats_prec85["tp"]),
        "holdout_tn_at_best_prec85": int(holdout_stats_prec85["tn"]),
        "holdout_fp_at_best_prec85": int(holdout_stats_prec85["fp"]),
        "holdout_fn_at_best_prec85": int(holdout_stats_prec85["fn"]),
    })


# ------------------------------------------------------------
# Display and Save Results
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows).sort_values("fraction").reset_index(drop=True)
history_df = pd.DataFrame(history_rows).sort_values(["fraction", "epoch"]).reset_index(drop=True)

print("\nFEW-SHOT SUMMARY TABLE (shared CA internal holdout)")
display(summary_df)

best_by_fraction = summary_df[[
    "fraction",
    "train_rows",
    "train_groups",
    "train_pos",
    "train_neg",
    "best_epoch",
    "holdout_auc",
    "holdout_ap",
    "holdout_f2_at_best_f2",
    "holdout_precision_at_best_prec85",
    "holdout_recall_at_best_prec85",
]].copy()

print("\nBEST-BY-FRACTION TABLE")
display(best_by_fraction)

# Text diagnosis of results
print("\n" + "=" * 72)
print("TEXT DIAGNOSIS")
print("=" * 72)
best_row = summary_df.sort_values("holdout_ap", ascending=False).iloc[0]
small_row = summary_df.sort_values("fraction", ascending=True).iloc[0]
large_row = summary_df.sort_values("fraction", ascending=False).iloc[0]

print(
    f"1) Best setting by shared-holdout AP: fraction={best_row['fraction']:.0%}, "
    f"holdout_auc={best_row['holdout_auc']:.4f}, holdout_ap={best_row['holdout_ap']:.4f}, "
    f"holdout_f2@bestF2={best_row['holdout_f2_at_best_f2']:.4f}."
)
print(
    f"2) Smallest-shot setting ({small_row['fraction']:.0%}) uses "
    f"{small_row['train_rows']} rows, {small_row['train_groups']} groups, "
    f"{small_row['train_pos']} pos / {small_row['train_neg']} neg, and gives "
    f"holdout_auc={small_row['holdout_auc']:.4f}, holdout_ap={small_row['holdout_ap']:.4f}, "
    f"holdout_prec@recall>=0.85={small_row['holdout_precision_at_best_prec85']:.4f}."
)
print(
    f"3) Largest-shot setting ({large_row['fraction']:.0%}) uses "
    f"{large_row['train_rows']} rows, {large_row['train_groups']} groups, "
    f"{large_row['train_pos']} pos / {large_row['train_neg']} neg, and gives "
    f"holdout_auc={large_row['holdout_auc']:.4f}, holdout_ap={large_row['holdout_ap']:.4f}, "
    f"holdout_prec@recall>=0.85={large_row['holdout_precision_at_best_prec85']:.4f}."
)
print(
    f"4) All fractions are trained for exactly {FS_NUM_EPOCHS} epochs; "
    f"best_epoch only means which epoch produced the best holdout precision under recall>=0.85, "
    f"not how long that fraction was trained."
)

# Save results to CSV
summary_df.to_csv(FS_SUMMARY_CSV, index=False)
history_df.to_csv(FS_HISTORY_CSV, index=False)

print("\nSaved CSVs:")
print(FS_SHARED_HOLDOUT_CSV)
print(FS_SUMMARY_CSV)
print(FS_HISTORY_CSV)


# ------------------------------------------------------------
# Visualize Learning Curve
# ------------------------------------------------------------

import matplotlib.pyplot as plt

plt.figure(figsize=(8.5, 5.2))
plt.plot(summary_df["train_rows"], summary_df["holdout_auc"], marker="o", label="Shared-holdout AUC")
plt.plot(summary_df["train_rows"], summary_df["holdout_ap"], marker="o", label="Shared-holdout AP")
plt.plot(summary_df["train_rows"], summary_df["holdout_f2_at_best_f2"], marker="o", label="Shared-holdout F2 @ best-F2")
plt.plot(summary_df["train_rows"], summary_df["holdout_recall_at_best_prec85"], marker="o", label="Shared-holdout Recall @ best-Prec(Recall>=0.85)")
plt.xscale("log")
plt.xlabel("Few-shot train rows (log scale)")
plt.ylabel("Metric")
plt.title("Few-shot fine-tune on shared CA internal holdout")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("\n" + "=" * 72)
print("FEW-SHOT TRANSFER LEARNING EXPERIMENT COMPLETE")
print("=" * 72)
print(f"Results show label efficiency: how much CA data is needed to adapt")
print(f"the Amazon-trained model to Central Asia archaeological sites.")
print("=" * 72)